# <center>Đồ án 1: Hồi quy tuyến tính</center>

# Thông tin sinh viên

- Họ và tên: Nguyễn Cao Bản
- MSSV: 24122029
- Lớp: 24TNT1

# Import

In [119]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn
# Import thư viện tại đây

# Đọc dữ liệu

In [ ]:
# Doc du lieu
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print('Kich thuoc train:', train.shape)
print('Kich thuoc test :', test.shape)

# Thiet lap cho bo Ames
# - train.csv co cot muc tieu SalePrice
# - test.csv cung co cot SalePrice de danh gia mo hinh
TARGET_COL = 'SalePrice'
ID_COL = 'Id'

X_train_full = train.drop(columns=[TARGET_COL, ID_COL], errors='ignore').copy()
y_train_full = train[TARGET_COL].copy()
X_test = test.drop(columns=[TARGET_COL, ID_COL], errors='ignore').copy()
y_test = test[TARGET_COL].copy()

# Dong bo thu tu cot giua tap train va test
X_test = X_test.reindex(columns=X_train_full.columns)

print('So dac trung dau vao:', X_train_full.shape[1])
print('Cot muc tieu:', TARGET_COL)

# Cài đặt hàm

In [121]:
# Cài đặt các hàm cần thiết ở đây

### Xây dựng lớp DataInfo (khám phá dữ liệu)

In [122]:
class DataInfo:
    """
    Class dùng để phân tích thông tin tổng quan của dataset 
    phục vụ Exploratory Data Analysis (EDA).
    """
    def __init__(self, train):
        self.train = train
        print("Tải tập train thành công!")
    def get_summary(self):
        '''Tổng quan dữ liệu'''
        print("Số lượng dòng:", self.train.shape[0])
        print("Số lượng cột:", self.train.shape[1])
        print("tên các cột:", self.train.columns.tolist())
    def describe_data(self):
        '''In ra Thống kê mô tả đặc trưng số'''
        print("\n--- Thống kê mô tả các đặc trưng số ---")
        print(self.train.describe())
    def head(self, n = 5):
        '''In ra n dòng đầu tiên'''
        print(self.train.head(n))
    def null_info(self):
        '''In ra thông tin dữ liệu null'''
        missing_values = self.train.isnull().sum()
        missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
        print("\nCác đặc trưng bị thiếu dữ liệu và số lượng null:")
        print(missing_values)
    def list_name_of_columns(self):
        '''In ra danh sách tên cột'''
        print("Danh sách tên cột: ", self.train.columns)
    def get_info(self):
        '''In ra thông ti dữ liệu'''
        self.train.info()
    def get_column_initial_info(self):
        '''In ra thông tin các đặc trưng'''
        pd.set_option('display.max_rows', None)
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', None)
        pd.set_option('display.max_colwidth', None)
        print("\nThông tin về các đặc trưng:")
        column_info = pd.DataFrame({
            'Column Name': self.train.columns,
            'Description': [
                "Identifies the type of dwelling involved in the sale",
                "Identifies the general zoning classification of the sale",
                "Linear feet of street connected to property",
                "Lot size in square feet",
                "Type of road access to property",
                "Type of alley access to property",
                "General shape of property",
                "Flatness of the property",
                "Type of utilities available",
                "Lot configuration",
                "Slope of property",
                "Physical locations within Ames city limits",
                "Proximity to various conditions",
                "Proximity to various conditions (if more than one is present)",
                "Type of dwelling",
                "Style of dwelling",
                "Rates the overall material and finish of the house",
                "Rates the overall condition of the house",
                "Original construction date",
                "Remodel date (same as construction date if no remodeling)",
                "Type of roof",
                "Roof material",
                "Exterior covering on house",
                "Exterior covering on house (if more than one material)",
                "Masonry veneer type",
                "Masonry veneer area in square feet",
                "Evaluates the quality of the material on the exterior",
                "Evaluates the present condition of the material on the exterior",
                "Type of foundation",
                "Evaluates the height of the basement",
                "Evaluates the general condition of the basement",
                "Refers to walkout or garden level walls",
                "Rating of basement finished area",
                "Type 1 finished square feet",
                "Rating of basement finished area (if multiple types)",
                "Type 2 finished square feet",
                "Unfinished square feet of basement area",
                "Total square feet of basement area",
                "Type of heating",
                "Heating quality and condition",
                "Central air conditioning",
                "Electrical system",
                "First Floor square feet",
                "Second floor square feet",
                "Low quality finished square feet",
                "Above grade living area square feet",
                "Basement full bathrooms",
                "Basement half bathrooms",
                "Full bathrooms above grade",
                "Half baths above grade",
                "Bedrooms above grade",
                "Kitchens above grade",
                "Kitchen quality",
                "Total rooms above grade",
                "Home functionality",
                "Number of fireplaces",
                "Fireplace quality",
                "Garage location",
                "Year garage was built",
                "Interior finish of the garage",
                "Size of garage in car capacity",
                "Size of garage in square feet",
                "Garage quality",
                "Garage condition",
                "Paved driveway",
                "Wood deck area in square feet",
                "Open porch area in square feet",
                "Enclosed porch area in square feet",
                "Three season porch area in square feet",
                "Screen porch area in square feet",
                "Pool area in square feet",
                "Pool quality",
                "Fence quality",
                "Miscellaneous feature not covered in other categories",
                "Value of miscellaneous feature",
                "Month Sold",
                "Year Sold",
                "Type of sale",
                "Condition of sale"
            ],
            'Data Type': self.train.dtypes.values,
            'Number of NaN': self.train.isna().sum().values,
            'Unique Values': self.train.nunique().values,
            'Most Frequent Value': self.train.mode().iloc[0].values,
        })
        return column_info
    def get_some_unique_values(self, k = 5):
        '''Lấy ra k dòng đầu tiên để hiển thị các unique values'''
        
        print(f"\nGiá trị unique của {k} cột đầu tiên:")
        lst_of_name_column = self.train.columns.tolist()
        k = min(k, len(lst_of_name_column))
        for i in range(k):
            print(f"{lst_of_name_column[i]}: ", self.train[lst_of_name_column[i]].unique())

    def unique_values(self):
        '''Thông tin về unique values'''
        object_columns = self.train.select_dtypes(include=['object']).columns
        numeric_columns = self.train.select_dtypes(include=['float64', 'int64']).columns

        object_columns_list = [(col, self.train[col].nunique()) for col in object_columns]
        numeric_columns_list = [(col, self.train[col].nunique()) for col in numeric_columns]

        print("Các cột thuộc kiểu dữ liệu Object và số lượng unique values: {}".format(len(object_columns_list)))
        print(object_columns_list)

        print("\nCác cột thuộc kiểu dữ liệu Numeric và số lượng unique values: {}".format(len(numeric_columns_list)))
        print(numeric_columns_list)
        self.numeric_columns = self.train.select_dtypes(include=['number']).columns
        self.object_columns = self.train.select_dtypes(include=['object']).columns
        return numeric_columns.tolist(), object_columns.tolist()


### Xây dựng lớp DataVisualization (Trực quan hóa dữ liệu)

In [123]:
class DataVisualization:
    """
    Class dùng để trực quan hóa dữ liệu phục vụ Exploratory Data Analysis (EDA)
    """
    def __init__(self, train):
        self.train = train
        self.test = test
        self.numeric_columns = self.train.select_dtypes(include = ['number']).columns
        self.object_columns = self.train.select_dtypes(include = ['object']).columns
        print("Tải tập train thành công")
    def plot_corr_matrix(self, width=12, height=8, k=5):
        """
        Vẽ ma trận tương quan (correlation matrix) cho k đặc trưng số đầu tiên.
        """
        corr_matrix = self.train.corr(numeric_only=True)
        # lấy k feature đầu
        corr_matrix = corr_matrix.iloc[:k, :k]
        plt.figure(figsize=(width, height))
        plt.imshow(corr_matrix, cmap='coolwarm', interpolation='none')
        plt.colorbar()
        plt.xticks(range(len(corr_matrix.columns)), 
                   corr_matrix.columns, 
                   rotation=90)
        plt.yticks(range(len(corr_matrix.columns)), 
                   corr_matrix.columns)
        plt.title(f"Ma trận tương quan của {k} đặc trưng số đầu tiên")
        for i in range(len(corr_matrix)):
            for j in range(len(corr_matrix)):
                plt.text(
                    j, i, 
                    f"{corr_matrix.iloc[i, j]:.2f}",
                    ha='center',
                    va='center',
                    color='black'
                )
        plt.tight_layout()
        plt.show()
    def plot_distribution_of_numeric_columns(self, bins = 30):
        """
        Vẽ histogram phân phối của các đặc trưng số trong dataset.
        """
        num_cols = self.numeric_columns
        num_cols_count = len(num_cols)
        n_cols = 3  
        n_rows = (num_cols_count + n_cols - 1) // n_cols  
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.5 * n_rows))
        axes = axes.flatten() 
        for i, col in enumerate(num_cols):
            axes[i].hist(self.train[col], bins=bins, color='skyblue', edgecolor='black', alpha=0.7)
            axes[i].set_title(f'Histogram of {col}')
            axes[i].set_xlabel(col)
            axes[i].set_ylabel('Frequency')
            axes[i].grid(axis='y', linestyle='--', alpha=0.7)
        for j in range(i + 1, len(axes)):
            fig.delaxes(axes[j])
        plt.tight_layout()
        plt.show()
    def box_plot_for_object_columns(self, target="SalePrice"):
        """
        Vẽ boxplot giữa các biến categorical và SalePrice
        """
        for col in self.object_columns:
            plt.figure(figsize=(12, 4))
            categories = self.train[col].dropna().unique()
            groups = [
                self.train.loc[self.train[col] == category, target]
                for category in categories
            ]
            plt.boxplot(
                groups,
                patch_artist=True,
                tick_labels=categories
            )
            plt.title(f'Box Plot: {target} by {col}')
            plt.xlabel(col)
            plt.ylabel(target)
            # rotate label nếu nhiều category
            if len(categories) > 6:
                plt.xticks(rotation=60)
            else:
                plt.xticks(rotation=0)
            plt.tight_layout()
            plt.show()
    def scatter_plot_for_numeric_columns(self, target="SalePrice"):
        """
        Vẽ scatter plot giữa numeric features và SalePrice
        """

        cols = [col for col in self.numeric_columns if col != target]

        n_plots = len(cols)
        n_cols = 3
        n_rows = (n_plots + n_cols - 1) // n_cols

        fig, axes = plt.subplots(
            n_rows, 
            n_cols, 
            figsize=(5 * n_cols, 4 * n_rows)
        )

        axes = axes.flatten()

        for i, col in enumerate(cols):

            axes[i].scatter(
                self.train[col],
                self.train[target],
                alpha=0.5
            )

            axes[i].set_title(f'{target} vs {col}')
            axes[i].set_xlabel(col)
            axes[i].set_ylabel(target)

            axes[i].grid(
                True,
                linestyle='--',
                alpha=0.5
            )

        # remove subplot thừa
        for j in range(i + 1, len(axes)):
            fig.delaxes(axes[j])

        plt.tight_layout()
        plt.show()
    def scatter_top_corr(self, target="SalePrice", k=9):
        """Lọc k top tương quan mạnh với SalePrice"""
        corr = self.train.corr(numeric_only=True)[target]\
            .abs()\
            .sort_values(ascending=False)[1:k+1]
        cols = corr.index
        n_cols = 3
        n_rows = (len(cols) + n_cols - 1) // n_cols
        fig, axes = plt.subplots(
            n_rows, 
            n_cols, 
            figsize=(5 * n_cols, 4 * n_rows)
        )
        axes = axes.flatten()
        for i, col in enumerate(cols):
            axes[i].scatter(
                self.train[col],
                self.train[target],
                alpha=0.5
            )
            axes[i].set_title(f'{target} vs {col}')
        plt.tight_layout()
        plt.show()
    def plot_target_distribution(self, target="SalePrice", bins=50):
        """
        Vẽ biểu đồ phân phối của biến mục tiêu (target)
        """
        plt.figure(figsize=(10, 5))
        sn.histplot(
            self.train[target],
            bins=bins,
            kde=True
        )
        plt.title(f"Phân phối của {target}")
        plt.xlabel(target)

        plt.tight_layout()
        plt.show()
    def plot_missing_values(self):
        """
        Vẽ biểu đồ thể hiện số lượng dữ liệu bị thiếu (missing values)
        """
        missing = self.train.isnull().sum()
        missing = missing[missing > 0]\
                    .sort_values(ascending=True)
        plt.figure(figsize=(10, 8))
        plt.barh(
            missing.index,
            missing.values
        )
        plt.title("Missing Values Distribution")
        plt.grid(
            True,
            linestyle='--',
            alpha=0.3
        )
        plt.tight_layout()
        plt.show()
    def plot_heatmap_top_k(self, target="SalePrice", k=10, width=10, height=8):
        """
        Vẽ Heatmap cho Top k đặc trưng có tương quan mạnh nhất với biến mục tiêu.
        Giúp kiểm tra tương quan chéo giữa các biến độc lập.
        """
        # 1. Tìm Top k đặc trưng có trị tuyệt đối tương quan cao nhất với target
        correlations = self.train.corr(numeric_only=True)[target].abs().sort_values(ascending=False)
        top_features = correlations.head(k + 1).index # +1 vì bao gồm cả chính nó (SalePrice)
        
        # 2. Trích xuất ma trận tương quan chỉ cho các biến này
        top_corr_matrix = self.train[top_features].corr()
        
        # 3. Vẽ biểu đồ
        plt.figure(figsize=(width, height))
        
        # Sử dụng seaborn (sn) để vẽ heatmap chuyên nghiệp hơn
        # annot=True để hiện số, fmt='.2f' làm tròn 2 chữ số
        sn.heatmap(top_corr_matrix, 
                   annot=True, 
                   cmap='RdYlGn', # Màu đỏ (âm) -> Vàng (0) -> Xanh (dương)
                   fmt='.2f', 
                   linewidths=0.5)
        
        plt.title(f"Heatmap: Top {k} Features Correlated with {target}")
        plt.tight_layout()
        plt.show()

In [124]:
def plot_regression_analysis(y_true, y_pred, option_name):
    """
    Gộp 3 biểu đồ phân tích hồi quy vào 1 hàng ngang.
    
    1. Actual vs Predicted: Kiểm tra độ chính xác tổng quát.
    2. Residual Plot: Kiểm tra tính ngẫu nhiên của sai số (Homoscedasticity).
    3. Error Distribution: Kiểm tra tính phân phối chuẩn của sai số.
    """
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle(f'Phân tích kết quả dự báo - {option_name}', fontsize=16, fontweight='bold')
    sn.scatterplot(x=y_true, y=y_pred, alpha=0.5, ax=axes[0])
    line_coords = [y_true.min(), y_true.max()]
    axes[0].plot(line_coords, line_coords, color='red', linestyle='--', lw=2)
    axes[0].set_title('Actual vs Predicted')
    axes[0].set_xlabel('Giá thực tế')
    axes[0].set_ylabel('Giá dự báo')
    axes[0].grid(True, alpha=0.3)
    residuals = y_true - y_pred
    sn.scatterplot(x=y_pred, y=residuals, alpha=0.5, ax=axes[1])
    axes[1].axhline(y=0, color='black', linestyle='--', lw=2)
    axes[1].set_title('Residual Plot (Sai số)')
    axes[1].set_xlabel('Giá trị dự báo')
    axes[1].set_ylabel('Sai số')
    axes[1].grid(True, alpha=0.3)
    sn.histplot(residuals, kde=True, color='purple', ax=axes[2])
    axes[2].set_title('Phân phối sai số')
    axes[2].set_xlabel('Giá trị sai số')
    axes[2].set_ylabel('Tần suất')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()


### Xây dựng lớp CleanData (Làm sạch dữ liệu, xử lí null,... )

In [125]:
class CleanData:
    """
    Xử lý giá trị thiếu (Missing Values) cho tập dữ liệu huấn luyện và kiểm thử.

    Lớp này thực hiện quy trình làm sạch dữ liệu dựa trên kiến thức chuyên môn (Domain Knowledge) 
    về dữ liệu nhà đất (Ames Housing). Thay vì chỉ điền giá trị trung bình/yếu vị một cách máy móc, 
    nó áp dụng các logic ràng buộc giữa các cột để đảm bảo tính nhất quán của dữ liệu.

    Cơ chế hoạt động:
    1. Lưu giữ các giá trị thống kê (Mode, Median) từ tập Train để áp dụng đồng nhất lên tập Test.
    2. Sử dụng 'Masking': Nếu diện tích tầng hầm/garage bằng 0, các đặc tính định tính 
       tương ứng sẽ được điền là 'None'.
    3. Xử lý các trường hợp đặc biệt như 'GarageYrBlt' dựa trên năm xây dựng nhà.

    Attributes:
        train_raw (pd.DataFrame): Bản sao của dữ liệu huấn luyện gốc (chưa xử lý).
        fill_values (dict): Từ điển lưu trữ giá trị dùng để điền (Mode/Median) cho từng cột, 
                           đảm bảo tập Test không bị rò rỉ dữ liệu (Data Leakage).
    """
    def __init__(self):
        self.train_raw = None
        self.fill_values = {}
    def transform(self, train, isXTest=False):
        """
        Thực hiện quy trình làm sạch và điền giá trị thiếu cho DataFrame.

        Quy trình cụ thể:
        - Điền 0 cho các cột diện tích/số lượng quan trọng (MasVnrArea, GarageCars,...).
        - Xử lý Basement: Nếu 'TotalBsmtSF' == 0, điền 'None' cho các cột loại tầng hầm.
        - Xử lý Garage: Nếu không có xe/diện tích garage == 0, điền 'None' cho chất lượng garage.
        - Xử lý năm xây Garage: Điền bằng 'YearBuilt' nếu không có thông tin năm xây garage.
        - Quét toàn bộ: Điền Mode (Yếu vị) cho biến phân loại và Median (Trung vị) cho biến số 
          dựa trên giá trị đã học được từ tập Train.

        Args:
            train (pd.DataFrame): DataFrame cần được xử lý (Train hoặc Test).
            isXTest (bool): Cờ đánh dấu dữ liệu đầu vào. 
                           - Nếu False: Học giá trị Mode/Median từ dữ liệu này.
                           - Nếu True: Sử dụng giá trị đã học từ trước để điền (tránh rò rỉ dữ liệu).

        Returns:
            pd.DataFrame: DataFrame mới đã được làm sạch hoàn toàn giá trị thiếu.
        """
        train = train.copy()
        # lưu train gốc
        if not isXTest:
            self.train_raw = train.copy()
        # Điền 0 cho các cột số dùng làm mask
        cols_to_fix_zero = ['TotalBsmtSF', 'GarageArea', 'GarageCars', 'MasVnrArea']
        for col in cols_to_fix_zero:
            if col in train.columns:
                train[col] = train[col].fillna(0)
        # Basement
        bsmt_cols = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']
        for col in bsmt_cols:
            if col in train.columns:
                mask = train['TotalBsmtSF'] == 0
                train.loc[mask, col] = train.loc[mask, col].fillna('None')
                if not isXTest:
                    self.fill_values[col] = train[col].mode()[0]
                train[col] = train[col].fillna(self.fill_values[col])
        # Garage
        train['GarageYrBlt'] = train['GarageYrBlt'].fillna(train['YearBuilt'])
        gara_cols = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']
        for col in gara_cols:
            if col in train.columns:
                mask = ((train['GarageArea'] == 0) & (train['GarageCars'] == 0))
                train.loc[mask, col] = train.loc[mask, col].fillna('None')
                if not isXTest:
                    self.fill_values[col] = train[col].mode()[0]
                train[col] = train[col].fillna(self.fill_values[col])
        # Others
        others = ['Alley', 'FireplaceQu', 'PoolQC', 'Fence', 'MiscFeature']
        for col in others:
            if col in train.columns:
                train[col] = train[col].fillna('None')
        # Quét lại categorical
        obj_cols = train.select_dtypes(include='object').columns
        for col in obj_cols:
            if not isXTest:
                if col not in self.fill_values:
                    self.fill_values[col] = train[col].mode()[0]
            # Điền giá trị yếu vị cho bất kỳ ô trống nào còn sót lại
            train[col] = train[col].fillna(self.fill_values.get(col, 'None'))
        # Quét lại numeric
        num_cols = train.select_dtypes(include=['int64', 'float64']).columns
        for col in num_cols:
            if not isXTest:
                if col not in self.fill_values:
                    self.fill_values[col] = train[col].median()
            # Điền median của Train cho bất kỳ ô số nào còn trống ở Test (thường là điền 0 hoặc median)
            train[col] = train[col].fillna(self.fill_values.get(col, 0))
        return train

### Xây dựng lớp EncodeCategorical (Mã hóa dữ liệu)

In [126]:
class EncodeCategorical:
    """
    Chuyển đổi các biến phân loại (Categorical Variables) thành định dạng số.

    Lớp này hỗ trợ hai chiến thuật mã hóa khác nhau để tối ưu hóa hiệu suất mô hình:
    1. Hybrid Encoding (Ordinal + One-hot): Sử dụng kiến thức chuyên môn để xếp hạng các biến 
       có tính thứ bậc (ví dụ: Chất lượng nhà: Excellent > Good > Average) và dùng One-hot 
       cho các biến định danh không có thứ tự.
    2. Full One-hot Encoding: Chuyển đổi tất cả các biến dạng chữ thành các cột nhị phân (0/1).

    Cơ chế Reindexing:
    Đảm bảo tính nhất quán giữa tập Train và tập Test. Nếu tập Test thiếu một giá trị 
    nào đó đã xuất hiện ở tập Train (hoặc ngược lại), phương thức transform sẽ tự động 
    cân bằng số lượng cột để tránh lỗi lệch ma trận (Dimension Mismatch).

    Attributes:
        qual_map (dict): Bảng ánh xạ cho các cột đánh giá chất lượng (Ex, Gd, TA, Fa, Po).
        qual_cols (list): Danh sách các cột áp dụng bảng ánh xạ chất lượng.
        columns (pd.Index): Danh sách tên các cột sau khi đã mã hóa (được học từ tập Train).
        isFullOnehot (bool): Trạng thái lưu phương pháp mã hóa được chọn.
    """
    def __init__(self):
        """Khởi tạo các bảng ánh xạ thứ bậc (Ordinal Maps) dựa trên Domain Knowledge."""
        self.qual_map = {'None':0, 'Po':1, 'Fa':2, 'TA':3, 'Gd':4, 'Ex':5}
        self.lotshape_map = {'Reg':3, 'IR1':2, 'IR2':1, 'IR3':0}
        self.bsmt_exposure = {'None':0, 'No':1, 'Mn':2, 'Av':3, 'Gd':4}
        self.bsmt_fin = {'None':0, 'Unf':1, 'LwQ':2, 'Rec':3, 'BLQ':4, 'ALQ':5, 'GLQ':6}
        self.garage_finish = {'None':0, 'Unf':1, 'RFn':2, 'Fin':3}
        self.fence_map = {'None':0, 'MnWw':1, 'GdWo':2, 'MnPrv':3, 'GdPrv':4}
        self.ulity_map = {'ELO':0, 'NoSeWa':1, 'NoSewr':2, 'AllPub':3}
        self.qual_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
                     'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']
        self.columns = None
        self.isFullOnehot = None
    def transform_with_half_onehot_ordinal(self, X_train_full):
        """
        Thực hiện mã hóa hỗn hợp: Ordinal cho biến thứ bậc và One-hot cho biến định danh.
        
        Phương pháp này giúp giảm số lượng cột sinh ra (so với Full One-hot) 
        và giữ lại thông tin về cường độ/thứ tự của các đặc tính chất lượng.
        """
        X_train_full = X_train_full.copy()
        for col in self.qual_cols:
            X_train_full[col] = X_train_full[col].map(self.qual_map)
        X_train_full['LotShape'] = X_train_full['LotShape'].map(self.lotshape_map)
        X_train_full['BsmtExposure'] = X_train_full['BsmtExposure'].map(self.bsmt_exposure)
        X_train_full['BsmtFinType1'] = X_train_full['BsmtFinType1'].map(self.bsmt_fin)
        X_train_full['BsmtFinType2'] = X_train_full['BsmtFinType2'].map(self.bsmt_fin)
        X_train_full['GarageFinish'] = X_train_full['GarageFinish'].map(self.garage_finish)
        X_train_full['Fence'] = X_train_full['Fence'].map(self.fence_map)
        X_train_full['Utilities'] = X_train_full['Utilities'].map(self.ulity_map)
        X_train_full = pd.get_dummies(X_train_full, columns=X_train_full.select_dtypes(include='object').columns, 
                                      dtype = 'int64')
        return X_train_full
    def transform_with_full_onehot(self, X_train_full):
        """
        Mã hóa One-hot toàn bộ các cột có kiểu dữ liệu là 'object'.
        
        Phù hợp khi không muốn giả định bất kỳ thứ tự nào giữa các giá trị phân loại.
        """
        X_train_full = X_train_full.copy()
        X_train_full = pd.get_dummies(X_train_full, columns=X_train_full.select_dtypes(include='object').columns, 
                                      dtype = 'int64')
        return X_train_full
    def fit_transform(self, X, isFullOnehot=False):
        """
        Học cấu trúc cột từ tập huấn luyện và thực hiện biến đổi.

        Args:
            X (pd.DataFrame): Dữ liệu huấn luyện.
            isFullOnehot (bool): Lựa chọn phương pháp mã hóa.

        Returns:
            pd.DataFrame: Dữ liệu đã mã hóa hoàn toàn sang số.
        """
        """Học danh sách cột sau khi mã hóa và trả về dữ liệu đã biến đổi."""
        self.isFullOnehot = isFullOnehot
        if not isFullOnehot:
            X = self.transform_with_half_onehot_ordinal(X)
        else:
            X = self.transform_with_full_onehot(X)
        self.columns = X.columns
        return X
    def transform(self, X):
        """
        Áp dụng mã hóa lên dữ liệu mới (Test) dựa trên danh sách cột đã học.

        Sử dụng reindex để đảm bảo tập dữ liệu mới có chính xác các cột 
        giống như tập Train, điền 0 cho các cột thiếu.

        Args:
            X (pd.DataFrame): Dữ liệu mới cần biến đổi.

        Returns:
            pd.DataFrame: Dữ liệu đã mã hóa đồng nhất với tập Train.
        """
        if not self.isFullOnehot:
            X = self.transform_with_half_onehot_ordinal(X)
        else:
            X = self.transform_with_full_onehot(X)
        X = X.reindex(
            columns=self.columns,
            fill_value=0
        )
        return X


### Xây dựng lớp Denoise (Xử lý đa cộng tuyến và biến dư thừa)


In [127]:
import pandas as pd
import numpy as np

class DenoiseData:
    """
    Xử lý nhiễu và tinh gọn tập dữ liệu thông qua lọc tương quan và tần suất.

    Class này giúp tối ưu hóa mô hình bằng cách:
    1. Loại bỏ các biến có phương sai thấp (Imbalanced Features): Các cột mà một giá trị 
       chiếm đa số (> threshold_freq), khiến chúng gần như là hằng số và không có giá trị phân loại.
    2. Khử đa cộng tuyến (Multicollinearity): Loại bỏ một trong hai biến có tương quan 
       quá cao (> threshold_corr) để tránh làm nhiễu hệ số hồi quy.
    3. Xử lý Dummy Trap & Redundancy: Loại bỏ các cột 'None' dư thừa phát sinh từ 
       quá trình One-hot Encoding hoặc các biến nhị phân gây ra bẫy biến giả.

    Attributes:
        threshold_corr (float): Ngưỡng tương quan Pearson để loại bỏ biến (0 đến 1).
        threshold_freq (float): Ngưỡng tần suất tập trung của một giá trị (0 đến 1).
        cols_to_drop (list): Danh sách các tên cột sẽ bị loại bỏ sau khi fit.
    """    
    def __init__(self, threshold_corr=0.9, threshold_freq=0.95):
        """
        Khởi tạo bộ lọc nhiễu với các ngưỡng tùy chỉnh.

        Args:
            threshold_corr (float): Nếu |corr(A, B)| > ngưỡng này, biến B sẽ bị loại.
            threshold_freq (float): Nếu một giá trị chiếm > ngưỡng này, cột đó sẽ bị loại.
        """
        self.threshold_corr = threshold_corr
        self.threshold_freq = threshold_freq
        self.cols_to_drop = []

    def fit(self, train):
        """
        Phân tích tập dữ liệu huấn luyện để xác định các cột cần loại bỏ.

        Quy trình phân tích:
        - Quét tần suất giá trị (Value Counts).
        - Tính toán ma trận tương quan (Correlation Matrix) và lọc nửa trên (Upper Triangle).
        - Kết hợp với các logic loại bỏ biến giả (Dummy Trap) thủ công.

        Args:
            train (pd.DataFrame): Tập dữ liệu huấn luyện dùng để học các cột nhiễu.

        Returns:
            self: Trả về chính đối tượng đã xác định xong danh sách cols_to_drop.
        """
        train = train.copy()
        temp_drops = []
        # --- 1. Loại bỏ biến có tần suất quá cao (Imbalanced Features) ---
        # Nếu một giá trị xuất hiện > threshold_freq (ví dụ 95%), cột đó gần như là hằng số
        n_samples = len(train)
        for col in train.columns:
            # Lấy tần suất của giá trị xuất hiện nhiều nhất
            max_freq_ratio = train[col].value_counts(normalize=True).max()
            if max_freq_ratio > self.threshold_freq:
                temp_drops.append(col)
        # --- 2. Tự động tìm các cặp tương quan > threshold_corr ---
        corr_matrix = train.corr(numeric_only=True).abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        corr_drops = [column for column in upper.columns 
                      if any(upper[column] > self.threshold_corr)]
        temp_drops.extend(corr_drops)
        # --- 3. Logic Dummy Trap & None Columns (giữ lại code cũ của bạn) ---
        logic_drops = ['Street_Grvl', 'CentralAir_N', 'Utilities_NoSeWa']
        bsmt_none = ['BsmtCond_None', 'BsmtExposure_None', 'BsmtFinType1_None', 'BsmtFinType2_None']
        garage_none = ['GarageFinish_None', 'GarageQual_None', 'GarageCond_None']
        
        temp_drops.extend(logic_drops)
        temp_drops.extend(bsmt_none)
        temp_drops.extend(garage_none)
        # Loại bỏ trùng lặp và kiểm tra tồn tại trong columns
        self.cols_to_drop = list(set([c for c in temp_drops if c in train.columns]))
        return self
    def transform(self, train):
        """
        Loại bỏ các cột đã xác định khỏi DataFrame.

        Args:
            train (pd.DataFrame): DataFrame cần làm sạch (Train hoặc Test).

        Returns:
            pd.DataFrame: DataFrame mới đã được loại bỏ các cột nhiễu.
        """
        train = train.copy()
        train = train.drop(columns=self.cols_to_drop, errors='ignore')
        return train

### Xây dựng lớp FeatureCreator (Tạo đặc trưng)

In [128]:
class FeatureCreator:
    """
    Kiến tạo các đặc trưng mới dựa trên logic.

    Lớp này thực hiện tổng hợp các biến thành phần thành các biến có giá trị dự báo cao hơn,
    giúp mô hình giảm bớt độ phức tạp và nắm bắt được các đặc điểm quan trọng của ngôi nhà
    như: Tổng diện tích, Tổng số phòng tắm và Tuổi thọ công trình.

    Quy trình tạo đặc trưng:
    1. Bath Score: Gộp các loại phòng tắm (Full/Half) ở cả tầng hầm và tầng nổi 
       theo trọng số chuẩn (Half bath tính 0.5).
    2. Total Living Space (TotalSF): Tổng diện tích sàn sử dụng thực tế bao gồm 
       tầng hầm, tầng 1 và tầng 2.
    3. Property Age: Chuyển đổi các cột năm (YearBuilt, YrSold) thành khoảng cách 
       thời gian (năm tuổi) để mô hình xử lý dưới dạng dữ liệu liên tục.

    Attributes:
        drop_components (bool): Nếu True, loại bỏ các cột gốc sau khi đã gộp 
                               để giảm đa cộng tuyến (Multicollinearity).
    """
    def __init__(self, drop_components=True):
        """
        Khởi tạo bộ tạo đặc trưng.

        Args:
            drop_components (bool): Quyết định việc xóa bỏ các cột thành phần sau xử lý.
        """
        self.drop_components = drop_components

    def transform(self, X):
        """
        Biến đổi dữ liệu bằng cách tính toán các đặc trưng tổng hợp.

        Logic chi tiết:
        - TotalBath: Kết hợp FullBath và HalfBath (Bsmt + Above Grade).
        - TotalSF: Tổng hợp diện tích bề mặt (Basement + 1st Floor + 2nd Floor).
        - HouseAge/RemodAge: Lấy năm bán trừ đi năm xây/sửa, giới hạn giá trị tối thiểu là 0.

        Args:
            X (pd.DataFrame): DataFrame đầu vào chứa các cột diện tích và năm.

        Returns:
            pd.DataFrame: DataFrame mới đã được bổ sung các cột đặc trưng tổng hợp.
        """
        X = X.copy()
        # 1. Tính toán điểm số phòng tắm (Bath Score)
        X['TotalBath'] = (X['FullBath'] + (0.5 * X['HalfBath']) + 
                          X['BsmtFullBath'] + (0.5 * X['BsmtHalfBath']))
        # 2. Tổng diện tích sử dụng thực tế (Total Living Space)
        X['TotalSF'] = X['TotalBsmtSF'] + X['1stFlrSF'] + X['2ndFlrSF']
        # 3. Tính toán tuổi thọ (Đưa về dạng khoảng cách thời gian)
        X['HouseAge'] = (X['YrSold'] - X['YearBuilt']).clip(lower=0)
        X['RemodAge'] = (X['YrSold'] - X['YearRemodAdd']).clip(lower=0)
        # 4. Loại bỏ cột thành phần để giảm gánh nặng cho ma trận X
        if self.drop_components:
            cols_to_drop = ['FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath', 
                            'TotalBsmtSF', '1stFlrSF', '2ndFlrSF']
            X = X.drop(columns=[c for c in cols_to_drop if c in X.columns], errors='ignore')
        return X
    def fit(self, X, y=None):
        """Không thực hiện học gì từ dữ liệu, chỉ để đảm bảo tính tương thích Pipeline."""
        return self
    def fit_transform(self, X, y=None):
        """Thực hiện fit và transform liên tiếp."""
        return self.transform(X)

### Xây dựng lớp FeatureTransformer (Biến đổi đặc trưng)

In [129]:
class FeatureTransformer:
    """
    Biến đổi đặc trưng tự động dựa trên phân phối toán học và tính phi tuyến.

    Lớp này thực hiện hai nhiệm vụ chính nhằm tối ưu hóa hiệu suất của mô hình hồi quy:
    1. Xử lý độ lệch (Skewness Correction): Sử dụng hàm log(1+x) để đưa các đặc trưng 
       bị lệch phải nặng (Right-skewed) về phân phối gần chuẩn (Normal Distribution). 
       Điều này giúp mô hình không bị chi phối quá mức bởi các giá trị ngoại lai (Outliers).
    2. Tạo đặc trưng đa thức (Polynomial Features): Tạo ra các biến bậc 2 cho những 
       đặc trưng quan trọng nhưng có mối quan hệ phi tuyến với giá nhà (ví dụ: OverallQual).

    Cơ chế hoạt động:
    - fit: Tính toán độ lệch (Skewness) của từng cột số và xác định danh sách các cột 
           vượt ngưỡng skew_threshold.
    - transform: Áp dụng phép biến đổi Log và lũy thừa dựa trên danh sách đã học.

    Attributes:
        skew_threshold (float): Ngưỡng độ lệch để quyết định có biến đổi Log hay không. 
                               Giá trị tuyệt đối càng lớn, tiêu chuẩn lọc càng gắt.
        poly_features (list): Danh sách các cột muốn tạo biến bậc 2 (bình phương).
        skewed_cols (list): Danh sách các tên cột bị lệch được xác định sau khi fit.
    """
    def __init__(self, skew_threshold=0.75, poly_features=['OverallQual']):
        """
        Khởi tạo bộ biến đổi đặc trưng.

        Args:
            skew_threshold (float): Ngưỡng độ lệch (mặc định 0.75).
            poly_features (list): Các cột để tạo biến bậc 2 (mặc định ['OverallQual']).
        """
        self.skew_threshold = skew_threshold
        self.poly_features = poly_features
        self.skewed_cols = []
    def fit(self, X, y=None):
        """
        Xác định các cột số có phân phối bị lệch nặng.

        Quy trình:
        - Lọc các cột kiểu số (numeric).
        - Loại bỏ các cột định danh hoặc cột có tính thứ bậc cố định (MSSubClass, v.v.).
        - Tính toán Skewness và lưu các cột có |skew| > skew_threshold.

        Args:
            X (pd.DataFrame): Dữ liệu huấn luyện.
            y: Không sử dụng, được giữ lại để tương thích với Pipeline.

        Returns:
            self: Trả về chính đối tượng sau khi học được danh sách skewed_cols.
        """
        # Xác định các cột số bị lệch phải nặng
        numeric_cols = X.select_dtypes(include=[np.number]).columns
        # Loại bỏ các cột định danh hoặc nhị phân (nếu có)
        exclude_cols = ['MSSubClass', 'OverallQual', 'OverallCond']
        cols_to_check = [c for c in numeric_cols if c not in exclude_cols]
        
        skewness = X[cols_to_check].skew().sort_values(ascending=False)
        self.skewed_cols = skewness[abs(skewness) > self.skew_threshold].index.tolist()
        return self

    def transform(self, X):
        """
        Áp dụng biến đổi Log và tạo biến đa thức lên DataFrame.

        Logic chi tiết:
        - Sử dụng np.log1p (log(1+x)) để tránh lỗi log(0) cho các diện tích bằng 0.
        - Đổi tên cột sang 'Log_ColumnName' để tăng tính tường minh trong báo cáo.
        - Tạo cột 'ColumnName_Sq' cho các biến đa thức được chỉ định.

        Args:
            X (pd.DataFrame): DataFrame cần biến đổi.

        Returns:
            pd.DataFrame: DataFrame mới với các đặc trưng đã được biến đổi toán học.
        """
        X = X.copy()
        # 1. Tự động Log các đặc trưng bị lệch đã tìm thấy ở bước fit
        for col in self.skewed_cols:
            if col in X.columns:
                # log1p để xử lý các giá trị 0 (như diện tích hồ bơi, hiên nhà)
                X[col] = np.log1p(X[col])
                # Đổi tên cột để dễ theo dõi trong báo cáo
                X.rename(columns={col: f'Log_{col}'}, inplace=True)
        # 2. Tạo biến bậc 2 cho các đặc trưng quan trọng nhưng không lệch
        for col in self.poly_features:
            if col in X.columns:
                X[f'{col}_Sq'] = X[col] ** 2       
        return X
    def fit_transform(self, X, y=None):
        return self.fit(X).transform(X)

### Xây dựng lớp OutlierRemover (Xử lý ngoại lai)


In [130]:
class OutlierRemover:
    """
    Loại bỏ các điểm dữ liệu ngoại lai (Outliers) dựa trên phương pháp Interquartile Range (IQR).

    Lớp này giúp làm sạch dữ liệu huấn luyện bằng cách xác định các giá trị nằm ngoài 
    "hàng rào" thống kê. Việc loại bỏ các căn nhà có diện tích hoặc đặc tính quá khác biệt 
    giúp mô hình Gradient Descent hội tụ ổn định hơn và tránh bị nhiễu bởi các trường hợp đặc biệt.

    Công thức xác định ngưỡng (Boxplot rule):
    - Q1: Bách phân vị thứ 25.
    - Q3: Bách phân vị thứ 75.
    - IQR = Q3 - Q1.
    - Ngưỡng dưới (Lower bound) = Q1 - (factor * IQR).
    - Ngưỡng trên (Upper bound) = Q3 + (factor * IQR).

    Cơ chế hoạt động:
    - fit: Tính toán và lưu trữ ngưỡng trên/dưới cho các cột được chỉ định từ tập Train.
    - transform: Lọc bỏ các dòng dữ liệu vi phạm ngưỡng. Lưu ý: Chỉ nên áp dụng transform 
                 cho tập huấn luyện (Train), không nên xóa dòng ở tập kiểm thử (Test).

    Attributes:
        factor (float): Hệ số nhân với IQR (mặc định 1.5 - mức chuẩn của Boxplot). 
                       Hệ số càng cao, bộ lọc càng ít khắt khe.
        columns (list): Danh sách các cột số cần kiểm tra ngoại lai.
        lower_bounds (dict): Lưu trữ ngưỡng dưới cho từng cột sau khi fit.
        upper_bounds (dict): Lưu trữ ngưỡng trên cho từng cột sau khi fit.
    """
    def __init__(self, factor=1.5, columns=['GrLivArea', 'TotalSF', 'LotArea']):
        """
        Khởi tạo bộ loại bỏ điểm ngoại lai.

        Args:
            factor (float): Hệ số xác định độ rộng của hàng rào dữ liệu (thường là 1.5 hoặc 3.0).
            columns (list): Các đặc trưng quan trọng dễ xuất hiện điểm dị biệt.
        """
        self.factor = factor
        self.columns = columns
        self.lower_bounds = {}
        self.upper_bounds = {}

    def fit(self, X, y=None):
        """
        Tính toán ngưỡng IQR cho các cột mục tiêu dựa trên dữ liệu hiện tại.

        Args:
            X (pd.DataFrame): Dữ liệu dùng để xác định phân phối chuẩn của các đặc trưng.
            y: Không sử dụng, được giữ lại để tương thích với Pipeline.

        Returns:
            self: Trả về đối tượng đã xác định xong các ngưỡng chặn.
        """
        # Tính toán ngưỡng cho từng cột
        for col in self.columns:
            if col in X.columns:
                Q1 = X[col].quantile(0.25)
                Q3 = X[col].quantile(0.75)
                IQR = Q3 - Q1
                self.lower_bounds[col] = Q1 - self.factor * IQR
                self.upper_bounds[col] = Q3 + self.factor * IQR
        return self

    def transform(self, X, y=None):
        """
        Lọc bỏ các dòng chứa điểm ngoại lai khỏi DataFrame.

        Nếu có tham số y đi kèm, phương thức sẽ thực hiện lọc đồng bộ trên cả X và y 
        để đảm bảo tính toàn vẹn của cặp dữ liệu (Feature - Target).

        Args:
            X (pd.DataFrame): DataFrame cần làm sạch.
            y (pd.Series/np.array, optional): Vector mục tiêu tương ứng với X.

        Returns:
            X_clean (pd.DataFrame): DataFrame sau khi đã loại bỏ các dòng ngoại lai.
            y_clean (optional): Vector mục tiêu sau khi đã lọc đồng bộ với X.
        """
        X = X.copy()
        initial_shape = X.shape[0]
        
        # Tạo mask để giữ lại các dòng nằm trong khoảng cho phép
        mask = np.ones(X.shape[0], dtype=bool)
        for col in self.columns:
            if col in X.columns:
                col_mask = (X[col] >= self.lower_bounds[col]) & (X[col] <= self.upper_bounds[col])
                mask = mask & col_mask
        
        X_clean = X[mask]
        # Nếu có y đi kèm (khi dùng cho tập Train), phải lọc y tương ứng
        if y is not None:
            # Đảm bảo y là Series hoặc mảng có cùng index với X ban đầu
            if isinstance(y, (pd.Series, pd.DataFrame)):
                y_clean = y.iloc[mask.values if hasattr(mask, 'values') else mask]
            else:
                y_clean = y[mask]
            return X_clean, y_clean
        return X_clean
    def fit_transform(self, X, y=None):
        """Thực hiện tính toán ngưỡng và loại bỏ các điểm ngoại lai ngay lập tức."""
        return self.fit(X, y).transform(X, y)

### Công cụ chuẩn hóa dữ liệu

In [131]:
class StandardScaler:
    """
    Chuẩn hóa dữ liệu về phân phối chuẩn hóa (Z-score Scaling).

    Phương pháp này biến đổi dữ liệu sao cho mỗi đặc trưng có giá trị trung bình (Mean) bằng 0 
    và độ lệch chuẩn (Standard Deviation) bằng 1. 
    Công thức: z = (x - mean) / std

    Ưu điểm:
    - Rất quan trọng cho Gradient Descent vì nó giúp hàm mất mát có hình dạng tròn trịa hơn, 
      giúp bước nhảy (learning rate) ổn định theo mọi hướng.
    - Giữ lại thông tin về các điểm ngoại lai (Outliers) tốt hơn so với MinMaxScaler.
    - Phù hợp với các đặc trưng có phân phối gần chuẩn (Gaussian).

    Attributes:
        mean (pd.Series): Giá trị trung bình của từng cột tính từ tập Train.
        std (pd.Series): Độ lệch chuẩn của từng cột tính từ tập Train.
        columns (pd.Index): Danh sách các cột được dùng để fit.
    """
    def __init__(self):
        """Khởi tạo các tham số lưu trữ trung bình, độ lệch chuẩn và danh sách cột."""
        self.mean = None
        self.std = None
        self.columns = None
    def fit(self, X):
        """
        Tính toán Mean và Std từ tập huấn luyện để sử dụng cho việc biến đổi.

        Args:
            X (pd.DataFrame): Dữ liệu huấn luyện.
        """
        self.columns = X.columns
        self.mean = X.mean(axis=0)
        self.std = X.std(axis=0)
        self.std[self.std == 0] = 1
    def transform(self, X):
        """
        Áp dụng chuẩn hóa lên dữ liệu mới dựa trên tham số đã học.

        Args:
            X (pd.DataFrame): Dữ liệu cần chuẩn hóa.

        Returns:
            pd.DataFrame: Dữ liệu đã được đưa về Mean=0, Std=1.
        """
        X = X[self.columns]
        return (X - self.mean) / self.std
    def fit_transform(self, X):
        """Thực hiện tính toán và chuẩn hóa trong một bước duy nhất."""
        self.fit(X)
        return self.transform(X)

class MinMaxScaler:
    """
    Thực hiện chuẩn hóa dữ liệu về khoảng cố định [0, 1] (Min-Max Scaling).

    Phương pháp này ép các giá trị dữ liệu vào một khoảng xác định, thường là từ 0 đến 1.
    Công thức: x_scaled = (x - min) / (max - min)

    Ưu điểm:
    - Đảm bảo tất cả các đặc trưng có cùng một thang đo tuyệt đối.
    - Phù hợp khi bạn biết rõ biên giới hạn của dữ liệu hoặc khi thuật toán yêu cầu 
      đầu vào không âm (như một số mạng nơ-ron hoặc mô hình khoảng cách).
    - Nhược điểm: Rất nhạy cảm với các điểm ngoại lai (Outliers). Nếu có một giá trị 
      cực lớn, nó sẽ ép các giá trị còn lại về sát mức 0.

    Attributes:
        min (pd.Series): Giá trị nhỏ nhất của từng cột.
        range (pd.Series): Khoảng cách giữa Max và Min (Max - Min).
        columns (pd.Index): Danh sách các cột được dùng để fit.
    """
    def __init__(self):
        """Khởi tạo scaler với các tham số trống."""    
        self.min = None
        self.range = None
        self.columns = None

    def fit(self, X, y=None):
        """
        Tính toán Min và Max từ tập huấn luyện.

        Args:
            X (pd.DataFrame): Dữ liệu huấn luyện.
        """
        self.columns = X.columns
        self.min = X.min(axis=0)
        max_val = X.max(axis=0)
        range_val = max_val - self.min
        range_val[range_val == 0] = 1 # Tránh chia cho 0
        self.range = range_val
        return self

    def transform(self, X):
        """
        Áp dụng chuẩn hóa Min-Max lên dữ liệu mới.

        Args:
            X (pd.DataFrame): Dữ liệu cần đưa về khoảng [0, 1].

        Returns:
            pd.DataFrame: Dữ liệu đã biến đổi.
        """
        X_subset = X[self.columns]
        return (X_subset - self.min) / self.range

    def fit_transform(self, X, y=None):
        """Thực hiện tính toán và chuẩn hóa trong một lần gọi."""
        return self.fit(X).transform(X)

### Công cụ đánh giá mô hình

In [132]:
def mean_absolute_error(y_true, y_pred):
    '''Tính trung bình tuyệt đối'''
    return np.mean(np.abs(y_true - y_pred))

### Mô hình dự đoán

In [133]:
class LinearRegression:
    """
    Hồi quy tuyến tính sử dụng Phương trình chuẩn (Ordinary Least Squares - OLS).
    Attributes:
        coef_ (numpy.ndarray): Các hệ số hồi quy (weights) cho từng đặc trưng.
        intercept_ (float): Hệ số chặn (bias), giá trị của y khi tất cả X bằng 0.
    """
    def __init__(self):
        """Khởi tạo hệ số góc (coef_) và hệ số chặn (intercept_)."""
        self.coef_ = None
        self.intercept_ = None
    def fit(self, X, y):
        """
        Huấn luyện mô hình bằng cách giải phương trình chuẩn.

        Sử dụng ma trận giả nghịch đảo (Moore-Penrose pseudoinverse) thay vì nghịch đảo 
        thông thường để đảm bảo tính ổn định toán học ngay cả khi ma trận (X^T * X) 
        bị suy biến (không nghịch đảo được).

        Args:
            X (array-like): Ma trận đặc trưng đầu vào, hình dạng (n_samples, n_features).
            y (array-like): Vector mục tiêu, hình dạng (n_samples,).

        Returns:
            self: Trả về chính đối tượng đã được huấn luyện.
        """
        X = np.asarray(X)
        y = np.asarray(y)
        # Thêm cột 1 vào ma trận X để tính toán Intercept (bias term)
        X_b = np.c_[np.ones(X.shape[0]), X]
        # Tính toán ma trận hiệp phương sai của đặc trưng: A = X.T * X
        A = np.dot(X_b.T, X_b)
        # Tính toán vector tương quan đặc trưng-mục tiêu: b = X.T * y
        b = np.dot(X_b.T, y)
        # Giải hệ phương trình A * beta = b bằng giả nghịch đảo: beta = A+ * b
        beta = np.dot(np.linalg.pinv(A), b)
        # Tách biệt hệ số chặn và hệ số góc
        self.intercept_ = beta[0]
        self.coef_ = beta[1:]
    def predict(self, X):
        """Dự đoán giá trị mục tiêu dựa trên các hệ số đã học: y = Xw + b."""
        X = np.asarray(X)
        return X @ self.coef_ + self.intercept_
class GradientDescent:
    """
    Mô hình Hồi quy Tuyến tính tối ưu hóa bằng Mini-batch Gradient Descent.

    Lớp này triển khai thuật toán Gradient Descent nâng cao tích hợp các kỹ thuật:
    1. Mini-batch: Cập nhật trọng số dựa trên một nhóm mẫu dữ liệu nhỏ, cân bằng giữa
       Stochastic GD và Batch GD.
    2. Momentum: Sử dụng biến vận tốc (v) để tăng tốc hội tụ và vượt qua các điểm cực tiểu địa phương.
    3. Regularization: Hỗ trợ L1 (Lasso), L2 (Ridge) và Elastic Net để chống Overfitting.
    4. Gradient Clipping: Giới hạn độ lớn của Gradient để tránh hiện tượng bùng nổ gradient.

    Attributes:
        theta (numpy.ndarray): Ma trận trọng số đầy đủ (bao gồm cả bias).
        coef_ (numpy.ndarray): Các hệ số hồi quy cho các đặc trưng (weights).
        intercept_ (float): Hệ số chặn (bias).
        losses (list): Danh sách giá trị Mean Squared Error qua từng epoch.
    """
    def __init__(self, learning_rate=0.01, epochs=1000, batch_size=32, tol=1e-5, 
                 gamma=0.9, penalty=None, alpha=0.1, l1_ratio=0.5):
        """
        Khởi tạo các tham số cho quá trình tối ưu hóa.

        Args:
            learning_rate (float): Tốc độ học (step size). Mặc định 0.01.
            epochs (int): Số lần lặp tối đa qua toàn bộ tập dữ liệu. Mặc định 1000.
            batch_size (int): Kích thước của mỗi nhóm mẫu dữ liệu khi cập nhật. Mặc định 32.
            tol (float): Ngưỡng dừng sớm nếu sai số thay đổi ít hơn giá trị này. Mặc định 1e-5.
            gamma (float): Hệ số Momentum (thường từ 0.5 đến 0.9). Mặc định 0.9.
            penalty (str): Loại hình phạt ('l1', 'l2', 'elasticnet' hoặc None).
            alpha (float): Độ mạnh của Regularization. Giá trị càng lớn, mô hình càng bị kiểm soát chặt.
            l1_ratio (float): Tỉ lệ giữa L1 và L2 trong Elastic Net (chỉ dùng khi penalty='elasticnet').
        """
        self.lr = learning_rate
        self.gamma = gamma
        self.epochs = epochs
        self.batch_size = batch_size
        self.tol = tol
        self.penalty = penalty
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        
        self.theta = None
        self.coef_ = None
        self.intercept_ = None
        self.losses = []
    def fit(self, X, y):
        """
        Huấn luyện mô hình trên tập dữ liệu đầu vào.

        Quá trình bao gồm xáo trộn dữ liệu (shuffling), cập nhật trọng số theo từng batch,
        tính toán gradient kèm theo các thành phần regularization, và kiểm tra điều kiện dừng sớm.

        Args:
            X (array-like): Ma trận đặc trưng đầu vào, hình dạng (n_samples, n_features).
            y (array-like): Vector mục tiêu, hình dạng (n_samples,).

        Returns:
            self: Trả về chính đối tượng đã được huấn luyện.
        """
        X = np.asarray(X)
        y = np.asarray(y).reshape(-1, 1)
        n_samples, n_features = X.shape
        X_b = np.c_[np.ones(n_samples), X]
        self.theta = np.random.randn(n_features + 1, 1) * 0.01
        v = np.zeros_like(self.theta)
        for epoch in range(self.epochs):
            indices = np.random.permutation(n_samples)
            X_shuffled = X_b[indices]
            y_shuffled = y[indices]
            for i in range(0, n_samples, self.batch_size):
                xi = X_shuffled[i : i + self.batch_size]
                yi = y_shuffled[i : i + self.batch_size]
                # 1. Gradient của hàm MSE (gốc)
                gradient = (2 / len(xi)) * xi.T @ (xi @ self.theta - yi)
                # 2. Cộng thêm Gradient của Regularization (Không áp dụng cho Bias - theta[0])
                reg_gradient = np.zeros_like(self.theta)
                # Tách theta_reg (không lấy bias) để tính toán
                theta_no_bias = self.theta.copy()
                theta_no_bias[0] = 0 
                if self.penalty == 'l2':
                    # Đạo hàm Ridge: alpha * 2 * theta
                    reg_gradient = 2 * self.alpha * theta_no_bias
                elif self.penalty == 'l1':
                    # Đạo hàm Lasso: alpha * sign(theta)
                    reg_gradient = self.alpha * np.sign(theta_no_bias)
                elif self.penalty == 'elasticnet':
                    # Kết hợp cả hai
                    l1 = self.l1_ratio * self.alpha * np.sign(theta_no_bias)
                    l2 = (1 - self.l1_ratio) * self.alpha * 2 * theta_no_bias
                    reg_gradient = l1 + l2
                gradient += reg_gradient
                # 3. Clip Gradient (Giữ nguyên logic của bạn)
                max_norm = 1.0
                norm = np.linalg.norm(gradient)
                if norm > max_norm:
                    gradient *= (max_norm / norm)
                # 4. Cập nhật Momentum
                v = self.gamma * v + self.lr * gradient
                self.theta -= v
            # Theo dõi Loss (bao gồm cả phần phạt để đảm bảo hội tụ đúng)
            mse = np.mean((X_b @ self.theta - y) ** 2)
            self.losses.append(mse)
            if epoch > 0 and abs(self.losses[-2] - self.losses[-1]) < self.tol:
                break  
        self.intercept_ = self.theta[0][0]
        self.coef_ = self.theta[1:].flatten()
        return self
    def predict(self, X):
        """
        Dự đoán giá trị dựa trên các trọng số đã học.

        Args:
            X (array-like): Ma trận đặc trưng cần dự đoán.

        Returns:
            numpy.ndarray: Vector chứa các giá trị dự đoán.
        """
        X = np.asarray(X)
        return X @ self.coef_ + self.intercept_




### Giảm chiều dữ liệu


In [134]:
class PCA:  
    """
    Phân tích thành phần chính (Principal Component Analysis - PCA).

    PCA là một kỹ thuật giảm chiều dữ liệu không giám sát (Unsupervised), giúp chuyển đổi 
    một tập hợp các biến có khả năng tương quan thành một tập hợp các biến không tương quan 
    tuyến tính gọi là các thành phần chính (Principal Components).

    Cơ chế toán học (Eigen-decomposition):
    1. Chuẩn tâm dữ liệu (Centering): Đưa dữ liệu về gốc tọa độ (Mean = 0).
    2. Ma trận hiệp phương sai (Covariance Matrix): Tính toán mối quan hệ tuyến tính giữa các đặc trưng.
    3. Trị riêng (Eigenvalues) & Vectơ riêng (Eigenvectors): 
       - Vectơ riêng xác định hướng của các trục mới (thành phần chính).
       - Trị riêng xác định độ lớn của phương sai (lượng thông tin) mà trục đó nắm giữ.
    4. Chiếu dữ liệu (Projection): Ánh xạ dữ liệu gốc lên các trục có phương sai lớn nhất.

    Attributes:
        n_components (int hoặc float): 
            - Nếu < 1: Tỉ lệ tổng phương sai muốn giữ lại (ví dụ: 0.95 giữ lại 95% thông tin).
            - Nếu >= 1: Số lượng trục (thành phần chính) cố định muốn lấy.
        mean (np.array): Giá trị trung bình của các đặc trưng gốc (dùng để transform).
        components (np.array): Ma trận các vectơ riêng (các trục mới).
        explained_variance_ratio (np.array): Tỉ lệ thông tin mà mỗi PC nắm giữ.
        feature_names (pd.Index): Tên các cột ban đầu.
    """
    def __init__(self, n_components=0.95):
        """
        Khởi tạo bộ giảm chiều PCA.

        Args:
            n_components (float/int): Ngưỡng thông tin hoặc số lượng thành phần cần giữ.
        """
        self.mean = None
        self.n_components = n_components
        self.components = None
        self.explained_variance_ratio = None
        self.feature_names = None
        # n_components : int hoặc float
        # - Nếu là float (0 < n < 1): Tỉ lệ phương sai giải thích cần giữ lại.
        # - Nếu là int: Số lượng thành phần chính cố định cần lấy.
    def fit(self, X, y = None):
        """
        Học các không gian vectơ mới từ dữ liệu huấn luyện.

        Các bước thực hiện:
        - Chuyển đổi dữ liệu sang Numpy array.
        - Tính ma trận hiệp phương sai mẫu.
        - Giải bài toán trị riêng bằng np.linalg.eigh (tối ưu cho ma trận đối xứng).
        - Sắp xếp các thành phần theo mức độ quan trọng giảm dần.

        Args:
            X (pd.DataFrame): Ma trận đặc trưng gốc.
            y: Không sử dụng (PCA là học không giám sát).

        Returns:
            self: Đối tượng đã học xong ma trận chiếu components.
        """
        X = X.copy()
        self.feature_names = X.columns
        X = np.array(X).astype(float)
        self.mean = np.mean(X, axis = 0)
        X_centered = X - self.mean

        # Tính ma trận hiệp phương sai
        n_samples = X.shape[0]
        cov_matrix = np.dot(X_centered.T, X_centered) / (n_samples - 1)

        # Tính trị riêng và vecto riêng
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

        # sắp xếp trị riêng theo thứ tự giảm dần
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]

        # Tính tỉ lệ phương sai giải thích
        total_var = np.sum(eigenvalues)
        self.explained_variance_ratio = eigenvalues / total_var

        if self.n_components < 1:
            cumsumm = np.cumsum(self.explained_variance_ratio)
            k = np.argmax(cumsumm >= self.n_components) + 1
        else:
            k = int(self.n_components)
        self.components = eigenvectors[:, :k]
        return self
    def transform(self, X):
        """
        Ánh xạ dữ liệu từ không gian cũ (n chiều) sang không gian mới (k chiều).

        Args:
            X (pd.DataFrame): Dữ liệu gốc cần giảm chiều.

        Returns:
            pd.DataFrame: Dữ liệu mới với các cột PC1, PC2, ... PCk.
        """
        original_index = X.index 
        X_arr = np.array(X).astype(float)
        X_centered = X_arr - self.mean
        X_pca = np.dot(X_centered, self.components)
        cols = [f'PC{i+1}' for i in range(X_pca.shape[1])]
        return pd.DataFrame(X_pca, columns=cols, index=original_index)

    def fit_transform(self, X, y=None):
        """Thực hiện học và giảm chiều dữ liệu trong một lần gọi."""
        return self.fit(X).transform(X)




*Chú thích: Cần có docstrings cho các hàm.*

### Xây dựng lớp KFold

In [135]:
class KFold:
    """
    Thực hiện K-Fold Cross-Validation để đánh giá mô hình.
    Tích hợp StandardScaler bên trong mỗi fold để tránh rò rỉ dữ liệu.
    """
    def __init__(self, n_splits=5, shuffle=False, random_state=None, scaler = True):
        """
        Khởi tạo cấu hình K-Fold
        """
        self.n_splits = n_splits
        self.shuffle = shuffle
        self.random_state = random_state
        self.scaler = scaler
    def split_and_calculateMAE(self, X, y, model, option = None):
        """
        Chia dữ liệu thành K phần, huấn luyện và tính MAE trung bình.
        Trả về: Giá trị MAE trung bình của tất cả các folds.
        """
        n_samples = len(X)
        indices = np.arange(n_samples)
        scores = []
        if self.shuffle:
            np.random.seed(self.random_state)
            np.random.shuffle(indices)
        fold_size = n_samples // self.n_splits
        for i in range(self.n_splits):
            start = i * fold_size
            end = n_samples if i == self.n_splits - 1 else (i + 1) * fold_size
            test_idx = indices[start:end]
            train_idx = np.concatenate([indices[:start], indices[end:]])

            X_train_fold, X_val_fold = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
            y_train_fold, y_val_fold = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

            if self.scaler:
                fold_scaler = StandardScaler()

                X_train_fold = pd.DataFrame(fold_scaler.fit_transform(X_train_fold), columns=X.columns)
                X_val_fold = pd.DataFrame(fold_scaler.transform(X_val_fold), columns=X.columns)
            
            if option == "option2":

                pca = PCA(n_components=0.99)
                X_train_fold = pca.fit_transform(X_train_fold)
                X_val_fold = pca.transform(X_val_fold)    
            elif option == "option3":

                selector = GradientDescent(penalty='l1', alpha=50, epochs=100)
                selector.fit(X_train_fold, y_train_fold)
                relevant_indices = np.where(np.abs(selector.coef_) > 1e-4)[0]
                
                X_train_fold = X_train_fold.iloc[:, relevant_indices]
                X_val_fold = X_val_fold.iloc[:, relevant_indices]

            model.fit(X_train_fold, y_train_fold)
            preds = model.predict(X_val_fold)
            mae = mean_absolute_error(y_val_fold, preds)
            scores.append(mae)
        return np.mean(scores)

# Yêu cầu 1: Phân tích khám phá dữ liệu (1 điểm)

Thực hiện EDA chủ yếu trên tập huấn luyện.
Có thể đối chiếu thống kê cơ bản giữa tập huấn luyện và tập kiểm tra, nhưng không dùng tập kiểm tra để điều chỉnh mô hình.

In [136]:
# Phân tích khám phá dữ liệu thông qua thống kê và các biểu đồ
# Chỉ được phân tích trên tập huấn luyện

# Data Overview

Khảo sát tổng quan dataset:
- Shape
- Data types
- Missing values
- Unique values
- Summary statistics

In [ ]:
OV = DataInfo(X_train_full)

In [ ]:
# Thông tin tổng quan dữ liệu
OV.get_summary()

In [ ]:
cat = X_train_full.select_dtypes(include = ['object']).columns.tolist()
num = X_train_full.select_dtypes(include = ['number']).columns.tolist()
print(len(cat), len(num))

In [ ]:
# Xem 5 dòng đầu tiên
OV.head()

In [ ]:
# Thông tin kiểu dữ liệu
OV.get_info()


In [ ]:
# Thống kê mô tả đặc trưng số
OV.describe_data()

In [ ]:
# Kiểm tra dữ liệu thiếu
OV.null_info()

In [ ]:
# Kiểm tra unique values
OV.unique_values()

In [ ]:
# Thông tin chi tiết từng cột
OV.get_column_initial_info()

# Data Visualization

Trực quan hóa dữ liệu gồm:

- Target distribution
- Missing values
- Correlation matrix
- Scatter plots
- Box plots

In [ ]:
VIS = DataVisualization(train.drop(columns = ["Id"], errors = "ignore"))

In [ ]:
#Phân tích tương quan giữa các biến số  
VIS.plot_corr_matrix(k = 10)

In [ ]:
# Phân tích tương quan giữa các biến số với label
VIS.plot_heatmap_top_k('SalePrice')

In [ ]:
# Phân tích phân phối các biến kiểu numeric
VIS.plot_distribution_of_numeric_columns()

In [ ]:
#Phân tích phân phối biến SalePrice
VIS.plot_target_distribution()

In [ ]:
# Quan hệ giữa biến kiểu numeric và SalePrice
VIS.scatter_plot_for_numeric_columns()

In [ ]:
# Các biến tương quan mạnh nhất với SalePrice
VIS.scatter_top_corr()

In [ ]:
# Ảnh hưởng của biến categorical đến SalePrice
VIS.box_plot_for_object_columns()

In [ ]:
# Phân tích dữ liệu thiếu
VIS.plot_missing_values()

# Yêu cầu 2a: Xây dựng mô hình sử dụng toàn bộ đặc trưng đầu vào (2 điểm)

Huấn luyện một mô hình Linear Regression sử dụng toàn bộ đặc trưng sau khi tiền xử lý phù hợp.
Sau đó báo cáo MAE của mô hình trên tập kiểm tra `test.csv`.

In [155]:
# Phần code cho yêu cầu 2a
X_train_2a = X_train_full.copy()
y_train_2a = y_train_full.copy()
X_test_2a = X_test.copy()
y_test_2a = y_test.copy()

In [156]:
# Làm sạch dữ liệu
Clean = CleanData()
X_train_2a = Clean.transform(X_train_2a, isXTest = False)
X_test_2a = Clean.transform(X_test_2a, isXTest = True)



In [157]:
# encode
encode = EncodeCategorical()
X_train_2a = encode.fit_transform(X_train_2a, isFullOnehot = False)
X_test_2a = encode.transform(X_test_2a)

In [158]:
# Chuẩn hóa
scaler = StandardScaler()
scaler.fit(X_train_2a)
X_train_2a = scaler.transform(X_train_2a)
X_test_2a = scaler.transform(X_test_2a)

In [ ]:
# Xem lại heatmap tương quan sau khi xử lý dữ liệu
df_full = pd.concat([X_train_2a, y_train_2a], axis=1)
asd = DataVisualization(df_full)
asd.plot_heatmap_top_k()

In [160]:
# Huấn luyện mô hình
model = LinearRegression()
model.fit(X_train_2a, y_train_2a)
y_pred = model.predict(X_test_2a)

In [ ]:
# Bao cao MAE cho Yeu cau 2a tren tap test
print('# Luu y: test.csv da co nhan SalePrice, co the danh gia truc tiep tren tap test')
mae_test_2a = mean_absolute_error(y_test_2a, y_pred)

print(f"MAE trên tập test.csv: {mae_test_2a:.3f}")

In [ ]:
plot_regression_analysis(y_test_2a, y_pred, "Phần 2a")

### BẢNG CHÚ THÍCH BIẾN ####


    Tên gốc (Feature) Ký hiệu (Abbr)
           MSSubClass             MS
          LotFrontage             LO
              LotArea            LO1
          OverallQual             OV
          OverallCond            OV1
            YearBuilt             YE
         YearRemodAdd            YE1
           MasVnrArea             MA
           BsmtFinSF1             BS
           BsmtFinSF2            BS1
            BsmtUnfSF            BS2
          TotalBsmtSF             TO
             1stFlrSF             1S
             2ndFlrSF             2N
         LowQualFinSF            LO2
            GrLivArea             GR
         BsmtFullBath            BS3
         BsmtHalfBath            BS4
             FullBath             FU
             HalfBath             HA
         BedroomAbvGr             BE
         KitchenAbvGr             KI
         TotRmsAbvGrd            TO1
           Fireplaces             FI
          GarageYrBlt             GA
           GarageCars            GA1
           GarageArea            GA2
           WoodDeckSF             WO
          OpenPorchSF             OP
        EnclosedPorch             EN
            3SsnPorch             3S
          ScreenPorch             SC
             PoolArea             PO
              MiscVal             MI
               MoSold             MO
               YrSold             YR
     MSZoning_C (all)            MS1
          MSZoning_FV            MS2
          MSZoning_RH            MS3
          MSZoning_RL            MS4
          MSZoning_RM            MS5
          Street_Grvl             ST
          Street_Pave            ST1
           Alley_Grvl             AL
           Alley_None            AL1
           Alley_Pave            AL2
         LotShape_IR1            LO3
         LotShape_IR2            LO4
         LotShape_IR3            LO5
         LotShape_Reg            LO6
      LandContour_Bnk             LA
      LandContour_HLS            LA1
      LandContour_Low            LA2
      LandContour_Lvl            LA3
     Utilities_AllPub             UT
     Utilities_NoSeWa            UT1
     LotConfig_Corner            LO7
    LotConfig_CulDSac            LO8
        LotConfig_FR2            LO9
        LotConfig_FR3           LO10
     LotConfig_Inside           LO11
        LandSlope_Gtl            LA4
        LandSlope_Mod            LA5
        LandSlope_Sev            LA6
    Neighborhood_Blmngtn             NE
    Neighborhood_Blueste            NE1
      Neighborhood_BrDale            NE2
    Neighborhood_BrkSide            NE3
    Neighborhood_ClearCr            NE4
    Neighborhood_CollgCr            NE5
    Neighborhood_Crawfor            NE6
    Neighborhood_Edwards            NE7
    Neighborhood_Gilbert            NE8
      Neighborhood_IDOTRR            NE9
    Neighborhood_MeadowV           NE10
    Neighborhood_Mitchel           NE11
      Neighborhood_NAmes           NE12
    Neighborhood_NPkVill           NE13
      Neighborhood_NWAmes           NE14
    Neighborhood_NoRidge           NE15
    Neighborhood_NridgHt           NE16
    Neighborhood_OldTown           NE17
      Neighborhood_SWISU           NE18
      Neighborhood_Sawyer           NE19
    Neighborhood_SawyerW           NE20
    Neighborhood_Somerst           NE21
    Neighborhood_StoneBr           NE22
      Neighborhood_Timber           NE23
    Neighborhood_Veenker           NE24
        Condition1_Artery             CO
        Condition1_Feedr            CO1
          Condition1_Norm            CO2
          Condition1_PosA            CO3
          Condition1_PosN            CO4
          Condition1_RRAe            CO5
          Condition1_RRAn            CO6
          Condition1_RRNe            CO7
          Condition1_RRNn            CO8
        Condition2_Artery            CO9
        Condition2_Feedr           CO10
          Condition2_Norm           CO11
          Condition2_PosA           CO12
          Condition2_PosN           CO13
          Condition2_RRAe           CO14
          Condition2_RRAn           CO15
          Condition2_RRNn           CO16
            BldgType_1Fam             BL
          BldgType_2fmCon            BL1
          BldgType_Duplex            BL2
          BldgType_Twnhs            BL3
          BldgType_TwnhsE            BL4
        HouseStyle_1.5Fin             HO
        HouseStyle_1.5Unf            HO1
        HouseStyle_1Story            HO2
        HouseStyle_2.5Fin            HO3
        HouseStyle_2.5Unf            HO4
        HouseStyle_2Story            HO5
        HouseStyle_SFoyer            HO6
          HouseStyle_SLvl            HO7
          RoofStyle_Flat             RO
          RoofStyle_Gable            RO1
        RoofStyle_Gambrel            RO2
            RoofStyle_Hip            RO3
        RoofStyle_Mansard            RO4
          RoofStyle_Shed            RO5
        RoofMatl_ClyTile            RO6
        RoofMatl_CompShg            RO7
        RoofMatl_Membran            RO8
          RoofMatl_Metal            RO9
            RoofMatl_Roll           RO10
        RoofMatl_Tar&Grv           RO11
        RoofMatl_WdShake           RO12
        RoofMatl_WdShngl           RO13
      Exterior1st_AsbShng             EX
      Exterior1st_AsphShn            EX1
      Exterior1st_BrkComm            EX2
      Exterior1st_BrkFace            EX3
      Exterior1st_CBlock            EX4
      Exterior1st_CemntBd            EX5
      Exterior1st_HdBoard            EX6
      Exterior1st_ImStucc            EX7
      Exterior1st_MetalSd            EX8
      Exterior1st_Plywood            EX9
        Exterior1st_Stone           EX10
      Exterior1st_Stucco           EX11
      Exterior1st_VinylSd           EX12
      Exterior1st_Wd Sdng           EX13
      Exterior1st_WdShing           EX14
      Exterior2nd_AsbShng           EX15
      Exterior2nd_AsphShn           EX16
      Exterior2nd_Brk Cmn           EX17
      Exterior2nd_BrkFace           EX18
      Exterior2nd_CBlock           EX19
      Exterior2nd_CmentBd           EX20
      Exterior2nd_HdBoard           EX21
      Exterior2nd_ImStucc           EX22
      Exterior2nd_MetalSd           EX23
        Exterior2nd_Other           EX24
      Exterior2nd_Plywood           EX25
        Exterior2nd_Stone           EX26
      Exterior2nd_Stucco           EX27
      Exterior2nd_VinylSd           EX28
      Exterior2nd_Wd Sdng           EX29
      Exterior2nd_Wd Shng           EX30
        MasVnrType_BrkCmn            MA1
      MasVnrType_BrkFace            MA2
        MasVnrType_Stone            MA3
            ExterQual_Ex           EX31
            ExterQual_Fa           EX32
            ExterQual_Gd           EX33
            ExterQual_TA           EX34
            ExterCond_Ex           EX35
            ExterCond_Fa           EX36
            ExterCond_Gd           EX37
            ExterCond_Po           EX38
            ExterCond_TA           EX39
        Foundation_BrkTil             FO
        Foundation_CBlock            FO1
        Foundation_PConc            FO2
          Foundation_Slab            FO3
        Foundation_Stone            FO4
          Foundation_Wood            FO5
              BsmtQual_Ex            BS5
              BsmtQual_Fa            BS6
              BsmtQual_Gd            BS7
            BsmtQual_None            BS8
              BsmtQual_TA            BS9
              BsmtCond_Fa           BS10
              BsmtCond_Gd           BS11
            BsmtCond_None           BS12
              BsmtCond_Po           BS13
              BsmtCond_TA           BS14
          BsmtExposure_Av           BS15
          BsmtExposure_Gd           BS16
          BsmtExposure_Mn           BS17
          BsmtExposure_No           BS18
        BsmtExposure_None           BS19
        BsmtFinType1_ALQ           BS20
        BsmtFinType1_BLQ           BS21
        BsmtFinType1_GLQ           BS22
        BsmtFinType1_LwQ           BS23
        BsmtFinType1_None           BS24
        BsmtFinType1_Rec           BS25
        BsmtFinType1_Unf           BS26
        BsmtFinType2_ALQ           BS27
        BsmtFinType2_BLQ           BS28
        BsmtFinType2_GLQ           BS29
        BsmtFinType2_LwQ           BS30
        BsmtFinType2_None           BS31
        BsmtFinType2_Rec           BS32
        BsmtFinType2_Unf           BS33
            Heating_Floor             HE
            Heating_GasA            HE1
            Heating_GasW            HE2
            Heating_Grav            HE3
            Heating_OthW            HE4
            Heating_Wall            HE5
            HeatingQC_Ex            HE6
            HeatingQC_Fa            HE7
            HeatingQC_Gd            HE8
            HeatingQC_Po            HE9
            HeatingQC_TA           HE10
            CentralAir_N             CE
            CentralAir_Y            CE1
        Electrical_FuseA             EL
        Electrical_FuseF            EL1
        Electrical_FuseP            EL2
          Electrical_Mix            EL3
        Electrical_SBrkr            EL4
          KitchenQual_Ex            KI1
          KitchenQual_Fa            KI2
          KitchenQual_Gd            KI3
          KitchenQual_TA            KI4
          Functional_Maj1            FU1
          Functional_Maj2            FU2
          Functional_Min1            FU3
          Functional_Min2            FU4
          Functional_Mod            FU5
          Functional_Sev            FU6
          Functional_Typ            FU7
          FireplaceQu_Ex            FI1
          FireplaceQu_Fa            FI2
          FireplaceQu_Gd            FI3
        FireplaceQu_None            FI4
          FireplaceQu_Po            FI5
          FireplaceQu_TA            FI6
        GarageType_2Types            GA3
        GarageType_Attchd            GA4
      GarageType_Basment            GA5
      GarageType_BuiltIn            GA6
      GarageType_CarPort            GA7
        GarageType_Detchd            GA8
          GarageType_None            GA9
        GarageFinish_Fin           GA10
        GarageFinish_None           GA11
        GarageFinish_RFn           GA12
        GarageFinish_Unf           GA13
            GarageQual_Ex           GA14
            GarageQual_Fa           GA15
            GarageQual_Gd           GA16
          GarageQual_None           GA17
            GarageQual_Po           GA18
            GarageQual_TA           GA19
            GarageCond_Ex           GA20
            GarageCond_Fa           GA21
            GarageCond_Gd           GA22
          GarageCond_None           GA23
            GarageCond_Po           GA24
            GarageCond_TA           GA25
            PavedDrive_N             PA
            PavedDrive_P            PA1
            PavedDrive_Y            PA2
                PoolQC_Ex            PO1
                PoolQC_Fa            PO2
                PoolQC_Gd            PO3
              PoolQC_None            PO4
              Fence_GdPrv             FE
              Fence_GdWo            FE1
              Fence_MnPrv            FE2
              Fence_MnWw            FE3
              Fence_None            FE4
        MiscFeature_Gar2            MI1
        MiscFeature_None            MI2
        MiscFeature_Othr            MI3
        MiscFeature_Shed            MI4
        MiscFeature_TenC            MI5
            SaleType_COD             SA
            SaleType_CWD            SA1
            SaleType_Con            SA2
          SaleType_ConLD            SA3
          SaleType_ConLI            SA4
          SaleType_ConLw            SA5
            SaleType_New            SA6
            SaleType_Oth            SA7
              SaleType_WD            SA8
    SaleCondition_Abnorml            SA9
    SaleCondition_AdjLand           SA10
    SaleCondition_Alloca           SA11
    SaleCondition_Family           SA12
    SaleCondition_Normal           SA13
    SaleCondition_Partial           SA14

### Công thức

Công thức hồi quy cho mô hình 2a (làm tròn hệ số đến 3 chữ số thập phân).

$$\text{SalePrice} = 180921.196 - 2040.304 * (MS) + 948.653 * (LO) + 7067.236 * (LO1) + 9261.766 * (OV) + 6398.107 * (OV1) + 10003.435 * (YE) + 2236.767 * (YE1) + 2909.146 * (MA) + 7942.870 * (BS) + 1563.970 * (BS1) - 77.038 * (BS2) + 8755.275 * (TO) + 5831.378 * (1S) + 14143.916 * (2N) - 1641.933 * (LO2) + 15887.795 * (GR) + 802.118 * (BS3) - 111.415 * (BS4) + 2040.105 * (FU) + 952.288 * (HA) - 3055.931 * (BE) - 2970.997 * (KI) + 2852.913 * (TO1) + 3995.457 * (FI) - 1070.567 * (GA) + 2738.953 * (GA1) + 4105.896 * (GA2) + 1947.169 * (WO) + 20.085 * (OP) + 158.704 * (EN) + 944.598 * (3S) + 2014.049 * (SC) + 27401.904 * (PO) + 83.579 * (MI) - 1317.378 * (MO) - 781.158 * (YR) - 2026.780 * (MS1) + 1833.559 * (MS2) - 176.371 * (MS3) + 398.559 * (MS4) - 997.326 * (MS5) - 1067.473 * (ST) + 1067.473 * (ST1) + 129.377 * (AL) - 136.712 * (AL1) + 57.634 * (AL2) - 606.323 * (LO3) + 645.271 * (LO4) + 413.272 * (LO5) + 300.397 * (LO6) - 512.913 * (LA) + 905.124 * (LA1) - 2125.792 * (LA2) + 889.536 * (LA3) + 462.960 * (UT) - 462.960 * (UT1) + 36.084 * (LO7) + 2098.769 * (LO8) - 1305.350 * (LO9) - 927.456 * (LO10) - 557.336 * (LO11) + 123.670 * (LA4) + 1624.096 * (LA5) - 3861.786 * (LA6) + 637.083 * (NE) + 622.145 * (NE1) + 532.306 * (NE2) + 377.055 * (NE3) - 950.795 * (NE4) - 841.550 * (NE5) + 3540.403 * (NE6) - 3552.671 * (NE7) - 795.526 * (NE8) - 676.307 * (NE9) + 171.302 * (NE10) - 2381.213 * (NE11) - 3644.182 * (NE12) + 1703.814 * (NE13) - 2215.518 * (NE14) + 5554.310 * (NE15) + 5595.568 * (NE16) - 1686.718 * (NE17) - 160.136 * (NE18) - 810.638 * (NE19) + 928.790 * (NE20) + 1175.707 * (NE21) + 6165.862 * (NE22) - 389.144 * (NE23) + 672.726 * (NE24) - 1924.688 * (CO) - 830.615 * (CO1) + 1926.331 * (CO2) - 142.905 * (CO3) + 446.494 * (CO4) - 2244.765 * (CO5) + 315.282 * (CO6) - 527.386 * (CO7) + 49.088 * (CO8) + 969.901 * (CO9) + 1301.481 * (CO10) + 1579.362 * (CO11) + 1788.121 * (CO12) - 7819.465 * (CO13) - 2636.246 * (CO14) + 34.285 * (CO15) + 868.162 * (CO16) + 2555.965 * (BL) + 340.291 * (BL1) - 186.860 * (BL2) - 2155.152 * (BL3) - 2226.433 * (BL4) - 25.172 * (HO) + 1187.389 * (HO1) + 2563.064 * (HO2) - 1266.120 * (HO3) - 850.593 * (HO4) - 2944.644 * (HO5) + 113.726 * (HO6) + 558.205 * (HO7) - 922.667 * (RO) - 73.660 * (RO1) + 215.523 * (RO2) - 181.130 * (RO3) + 685.465 * (RO4) + 3325.977 * (RO5) - 14967.698 * (RO6) + 298.241 * (RO7) + 2572.526 * (RO8) + 1754.311 * (RO9) - 248.093 * (RO10) + 312.060 * (RO11) - 299.879 * (RO12) + 3688.520 * (RO13) + 1315.839 * (EX) - 306.952 * (EX1) + 290.118 * (EX2) + 3503.994 * (EX3) - 184.613 * (EX4) - 204.283 * (EX5) - 869.186 * (EX6) - 266.904 * (EX7) + 1757.010 * (EX8) - 796.978 * (EX9) + 323.355 * (EX10) + 500.852 * (EX11) - 1246.405 * (EX12) - 1022.895 * (EX13) + 201.492 * (EX14) - 1079.255 * (EX15) + 69.739 * (EX16) - 237.171 * (EX17) - 628.911 * (EX18) - 184.613 * (EX19) + 638.015 * (EX20) - 320.142 * (EX21) + 628.996 * (EX22) - 1219.593 * (EX23) - 754.770 * (EX24) - 794.744 * (EX25) - 1173.087 * (EX26) - 470.504 * (EX27) + 1618.643 * (EX28) + 883.473 * (EX29) - 601.814 * (EX30) - 704.719 * (MA1) - 477.506 * (MA2) + 753.128 * (MA3) + 3541.856 * (EX31) + 1150.666 * (EX32) - 1040.845 * (EX33) - 571.995 * (EX34) + 265.803 * (EX35) + 318.412 * (EX36) - 578.033 * (EX37) + 331.267 * (EX38) + 333.250 * (EX39) - 746.630 * (FO) + 77.847 * (FO1) + 741.852 * (FO2) - 1224.853 * (FO3) + 510.903 * (FO4) - 1326.255 * (FO5) + 3811.740 * (BS5) + 328.635 * (BS6) - 2092.616 * (BS7) + 445.336 * (BS8) - 276.146 * (BS9) - 445.690 * (BS10) - 502.962 * (BS11) + 445.336 * (BS12) + 2338.872 * (BS13) + 80.171 * (BS14) + 128.286 * (BS15) + 4276.253 * (BS16) - 849.237 * (BS17) - 2359.302 * (BS18) + 445.336 * (BS19) - 863.272 * (BS20) + 109.937 * (BS21) + 1355.123 * (BS22) - 1303.360 * (BS23) + 445.336 * (BS24) - 672.815 * (BS25) + 159.258 * (BS26) + 1033.941 * (BS27) - 596.528 * (BS28) + 681.932 * (BS29) - 913.300 * (BS30) + 445.336 * (BS31) - 206.855 * (BS32) + 97.156 * (BS33) - 191.673 * (HE) + 246.792 * (HE1) - 108.278 * (HE2) - 464.349 * (HE3) - 725.768 * (HE4) + 760.424 * (HE5) + 944.850 * (HE6) + 458.269 * (HE7) - 783.257 * (HE8) + 107.875 * (HE9) - 586.434 * (HE10) + 22.571 * (CE) - 22.571 * (CE1) + 340.343 * (EL) + 134.554 * (EL1) - 321.878 * (EL2) - 1007.196 * (EL3) - 217.064 * (EL4) + 5223.364 * (KI1) + 60.045 * (KI2) - 1583.893 * (KI3) - 1105.360 * (KI4) - 1075.805 * (FU1) - 733.584 * (FU2) - 528.483 * (FU3) - 313.074 * (FU4) - 1639.734 * (FU5) - 1347.960 * (FU6) + 1867.503 * (FU7) - 644.879 * (FI1) - 908.858 * (FI2) - 1084.941 * (FI3) + 1779.820 * (FI4) + 820.058 * (FI5) - 708.473 * (FI6) - 1390.647 * (GA3) - 743.643 * (GA4) + 249.953 * (GA5) - 427.884 * (GA6) + 272.914 * (GA7) + 911.023 * (GA8) + 454.022 * (GA9) + 356.064 * (GA10) + 454.022 * (GA11) - 756.244 * (GA12) + 175.757 * (GA13) + 5342.116 * (GA14) - 1247.258 * (GA15) - 151.898 * (GA16) + 454.022 * (GA17) - 1110.566 * (GA18) - 192.732 * (GA19) - 4155.052 * (GA20) - 146.689 * (GA21) - 180.362 * (GA22) + 454.022 * (GA23) + 330.548 * (GA24) + 219.836 * (GA25) + 135.770 * (PA) - 413.948 * (PA1) + 94.922 * (PA2) - 2866.484 * (PO1) - 8706.333 * (PO2) - 9209.199 * (PO3) + 12233.694 * (PO4) - 1544.095 * (FE) + 14.428 * (FE1) + 486.469 * (FE2) - 415.578 * (FE3) + 473.254 * (FE4) - 142.450 * (MI1) - 326.910 * (MI2) + 496.054 * (MI3) + 153.118 * (MI4) + 804.309 * (MI5) - 1253.475 * (SA) + 356.018 * (SA1) + 657.961 * (SA2) + 675.444 * (SA3) - 150.317 * (SA4) - 373.152 * (SA5) + 3826.275 * (SA6) + 51.496 * (SA7) - 2699.979 * (SA8) - 846.020 * (SA9) + 336.958 * (SA10) - 162.605 * (SA11) - 293.837 * (SA12) + 1355.245 * (SA13) - 979.792 * (SA14)$$

# Yêu cầu 2b: Xây dựng mô hình với một đặc trưng và tìm mô hình tốt nhất (2 điểm)

Huấn luyện một mô hình cho từng đặc trưng và so sánh bằng MAE trung bình theo k-fold cross-validation trên tập huấn luyện.
Sau khi chọn được đặc trưng tốt nhất, huấn luyện lại mô hình trên toàn bộ tập huấn luyện và báo cáo MAE trên tập kiểm tra `test.csv`.

Lưu ý: Chỉ xáo trộn dữ liệu một lần trước cross-validation và giữ cùng chính sách chia cho mọi đặc trưng ứng viên. Việc chọn đặc trưng phải dựa trên tập huấn luyện.

In [164]:
# Phan code cho Yeu cau 2b
# 1) Huan luyen cac mo hinh voi tung dac trung don le
# 2) Tinh MAE trung binh theo CV cho moi dac trung
# 3) Chon dac trung co MAE nho nhat
X_train_2b = X_train_full.copy()
y_train_2b = y_train_full.copy()
X_test_2b = X_test.copy()
y_test_2b = y_test.copy()


In [165]:
# Làm sạch dữ liệu
Clean = CleanData()
X_train_2b = Clean.transform(X_train_2b, isXTest = False)
X_test_2b = Clean.transform(X_test_2b, isXTest = True)

# encode
encode = EncodeCategorical()
X_train_2b = encode.fit_transform(X_train_2b, isFullOnehot = True)
X_test_2b = encode.transform(X_test_2b)

# Chuẩn hóa
# scaler = StandardScaler()
# scaler.fit(X_train_2b)
# X_train_2b = scaler.transform(X_train_2b)
# X_test_2b = scaler.transform(X_test_2b)


# Chọn mô hình
model = LinearRegression()

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42, scaler = True)
candidate_features = ['OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', 'FullBath']
cv_results = {}

for col in candidate_features:
    X_single = X_train_2b[[col]]
    y_single = y_train_2b
    avg_mae = kf.split_and_calculateMAE(X_single, y_single, LinearRegression())
    cv_results[col] = avg_mae
    print(f"Feature: {col:15} | Average CV MAE: {avg_mae:.4f}")
best_feature_name = min(cv_results, key=cv_results.get)
print(f"\n=> Đặc trưng tốt nhất xác định bởi K-Fold: {best_feature_name}")

In [167]:

X_final_train = X_train_2b[[best_feature_name]]
X_final_test = X_test_2b[[best_feature_name]]


final_scaler = StandardScaler()
X_train_final_scaled = final_scaler.fit_transform(X_final_train)
X_test_final_scaled = final_scaler.transform(X_final_test)

best_feature_model = LinearRegression()
best_feature_model.fit(X_train_final_scaled, y_train_2b)

y_pred_test_2b = best_feature_model.predict(X_test_final_scaled)

mae_test_2b = mean_absolute_error(y_test_2b, y_pred_test_2b)


In [ ]:
# Danh gia MAE cho best_feature_model tren tap test
print('# Sau khi chon dac trung tot nhat bang CV tren train, bao cao MAE tren test.csv')
print(f"Model sử dụng biến: {best_feature_name}")
print(f"MAE cuối cùng trên tập Test: {mae_test_2b:,.2f}")

In [ ]:
plot_regression_analysis(y_test_2b, y_pred_test_2b, "Phần 2b")

Công thức hồi quy cho mô hình 2b (đặc trưng tốt nhất, làm tròn đến 3 chữ số thập phân).

$$\text{SalePrice} = 180921.196 + 62837.558 * \text{OverallQual}$$

# Yêu cầu 2c: Tự thiết kế các mô hình đặc trưng và tìm mô hình tốt nhất (2 điểm)

Thiết kế ít nhất 3 phương án đặc trưng/mô hình khác nhau.
So sánh các phương án bằng k-fold cross-validation trên tập huấn luyện, sau đó huấn luyện lại mô hình tốt nhất trên toàn bộ tập huấn luyện và báo cáo MAE trên tập kiểm tra `test.csv`.

In [170]:
# Trinh bay toan bo code dung de thiet ke cac phuong an dac trung/mo hinh
X_train_2c = X_train_full.copy()
y_train_2c = y_train_full.copy()
X_test_2c = X_test.copy()
y_test_2c = y_test.copy()

Lưu ý: Chỉ xáo trộn dữ liệu một lần và đánh giá toàn bộ $m$ mô hình theo cùng thiết lập cross-validation.

In [171]:
# Phan code cho Yeu cau 2c
# 1) Dinh nghia m >= 3 phuong an dac trung/mo hinh
# 2) Danh gia tung phuong an bang k-fold CV
# 3) Chon phuong an co MAE trung binh thap nhat
def pipeline_for_option1(X_train_2c, X_test_2c, y_train_2c):
    """
    Pipeline cho phương án 1
    """
    X_train = X_train_2c.copy()
    X_test = X_test_2c.copy()
    y_train = y_train_2c.copy()
    # Làm sạch dữ liệu
    Clean = CleanData()
    X_train = Clean.transform(X_train, isXTest = False)
    X_test = Clean.transform(X_test, isXTest = True)

    # Tạo dữ liệu mới (kết hợp dữ liệu)
    FC = FeatureCreator(drop_components = True)
    X_train = FC.transform(X_train)
    X_test = FC.transform(X_test)

    # Loại bỏ outliers
    OR = OutlierRemover()
    X_train, y_train = OR.fit_transform(X_train, y_train)

    ## Biến đổi dữ liệu (log, bình phương,...)
    FT = FeatureTransformer()
    X_train = FT.fit_transform(X_train)
    X_test = FT.transform(X_test)

    # encode
    encode = EncodeCategorical()
    X_train = encode.fit_transform(X_train, isFullOnehot = True)
    X_test = encode.transform(X_test)

    # Loại nhiễu
    DD = DenoiseData()
    DD.fit(X_train)
    X_train = DD.transform(X_train)
    X_test = DD.transform(X_test)
    # Chuẩn hóa
    # scaler = StandardScaler()
    # scaler.fit(X_train)
    # X_train = scaler.transform(X_train)
    # X_test = scaler.transform(X_test)

    return X_train, X_test, y_train

def pipeline_for_option2(X_train_2c, X_test_2c, y_train_2c):
    """
    Pipeline cho phương án 2
    """
    X_train = X_train_2c.copy()
    X_test = X_test_2c.copy()
    y_train = y_train_2c.copy()

    # Làm sạch dữ liệu
    Clean = CleanData()
    X_train = Clean.transform(X_train, isXTest = False)
    X_test = Clean.transform(X_test, isXTest = True)

    # encode
    encode = EncodeCategorical()
    X_train = encode.fit_transform(X_train, isFullOnehot = True)
    X_test = encode.transform(X_test)
   

    # Chuẩn hóa
    # scaler = StandardScaler()
    # scaler.fit(X_train)
    # X_train = scaler.transform(X_train)
    # X_test = scaler.transform(X_test)

    # Giảm chiều dữ liệu
    # pca = PCA(n_components = 0.99)
    # pca.fit(X_train)
    # X_train = pca.transform(X_train)
    # X_test = pca.transform(X_test)

    return X_train, X_test, y_train

def pipeline_for_option3(X_train_3c, X_test_3c, y_train_3c):
    """
    Pipeline Phương án 3: Lọc đặc trưng bằng Lasso và chuẩn bị cho Gradient Descent.
    """
    X_train = X_train_3c.copy()
    X_test = X_test_3c.copy()
    y_train = y_train_3c.copy()

    # 1. Làm sạch dữ liệu
    Clean = CleanData()
    X_train = Clean.transform(X_train, isXTest=False)
    X_test = Clean.transform(X_test, isXTest=True)

    # encode
    encode = EncodeCategorical()
    X_train = encode.fit_transform(X_train, isFullOnehot=True)
    X_test = encode.transform(X_test)

    # Chuẩn hóa
    # scaler = StandardScaler()
    # X_train = scaler.fit_transform(X_train)
    # X_test = scaler.transform(X_test)

    # 4. Feature Selection (Lọc biến bằng Lasso GD)
    # selector = GradientDescent(penalty='l1', alpha=0.1, epochs=100)
    # selector.fit(X_train, y_train)
    
    # Giữ lại các cột có trọng số đáng kể (trị tuyệt đối > 1e-4)
    # relevant_indices = np.where(np.abs(selector.coef_) > 1e-4)[0]
    # X_train = X_train.iloc[:, relevant_indices]
    # X_test = X_test.iloc[:, relevant_indices]   

    return X_train, X_test, y_train


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=36, scaler = True)
X_train_opt1, X_test_opt1, y_train_opt1 = pipeline_for_option1(X_train_2c, X_test_2c, y_train_2c)
X_train_opt2, X_test_opt2, y_train_opt2 = pipeline_for_option2(X_train_2c, X_test_2c, y_train_2c)
X_train_opt3, X_test_opt3, y_train_opt3 = pipeline_for_option3(X_train_2c, X_test_2c, y_train_2c)

options = {
    "option1": (X_train_opt1, y_train_opt1, X_test_opt1, LinearRegression()),
    "option2": (X_train_opt2, y_train_opt2, X_test_opt2, LinearRegression()),
    "option3": (X_train_opt3, y_train_opt3, X_test_opt3, GradientDescent(learning_rate = 5, 
                                                                        epochs = 1000,
                                                                        batch_size = 64,
                                                                        alpha = 0.1,
                                                                        penalty = 'l2',
                                                                        ))
}
results = {}
for name, (X, y, _, model) in options.items():
    score = kf.split_and_calculateMAE(X, y, model, name)
    results[name] = score

if results:
    best_feature = min(results, key=results.get)
    print(f"\nPhương án tốt nhất: {best_feature} với MAE: {results[best_feature]:,.2f}")

In [ ]:
# Huan luyen lai my_best_model tren toan bo tap huan luyen

X_final_train, y_final_train, X_final_test, my_best_model = options[best_feature]
scaler = StandardScaler()
X_final_train = pd.DataFrame(scaler.fit_transform(X_final_train), columns=X_final_train.columns)
X_final_test = pd.DataFrame(scaler.transform(X_final_test), columns=X_final_train.columns)

if best_feature == "option2":
    pca = PCA(n_components=0.99)
    X_final_train = pca.fit_transform(X_final_train)
    X_final_test = pca.transform(X_final_test)
elif best_feature == "option3":
    selector = GradientDescent(penalty='l1', alpha=50, epochs=100)
    selector.fit(X_final_train, y_final_train)
    rel_idx = np.where(np.abs(selector.coef_) > 1e-4)[0]
    X_final_train = X_final_train.iloc[:, rel_idx]
    X_final_test = X_final_test.iloc[:, rel_idx]

my_best_model.fit(X_final_train, y_final_train)

y_pred_final = my_best_model.predict(X_final_test)
mae_final = mean_absolute_error(y_test_2c, y_pred_final)
print("phương án tối ưu nhất: ", {best_feature})
print(f"MAE cuối cùng trên tập Test: {mae_final:,.2f}")

In [ ]:
# Danh gia MAE cho my_best_model tren tap validation
print('# Co the so sanh MAE validation voi MAE CV cua phuong an duoc chon')
print(f"MAE CV: {results[best_feature]:,.2f}")
print(f"MAE Test: {mae_final:,.2f}")


In [ ]:
plot_regression_analysis(y_test_2c, y_pred_final, "option1")


### Bảng chú thích biến

    Tên gốc (Feature) Ký hiệu (Abbr)
           MSSubClass             MS
          LotFrontage             LO
              LotArea            LO1
          OverallQual             OV
          OverallCond            OV1
            YearBuilt             YE
         YearRemodAdd            YE1
       Log_MasVnrArea            LO2
           BsmtFinSF1             BS
       Log_BsmtFinSF2            LO3
        Log_BsmtUnfSF            LO4
            GrLivArea             GR
         BedroomAbvGr             BE
         TotRmsAbvGrd             TO
           Fireplaces             FI
          GarageYrBlt             GA
           GarageCars            GA1
           GarageArea            GA2
       Log_WoodDeckSF            LO5
      Log_OpenPorchSF            LO6
    Log_EnclosedPorch            LO7
      Log_ScreenPorch            LO8
               MoSold             MO
               YrSold             YR
            TotalBath            TO1
              TotalSF            TO2
          MSZoning_RL            MS1
          MSZoning_RM            MS2
           Alley_None             AL
         LotShape_IR1            LO9
      LandContour_Lvl             LA
     LotConfig_Corner           LO10
    LotConfig_CulDSac           LO11
     LotConfig_Inside           LO12
    Neighborhood_CollgCr             NE
    Neighborhood_Edwards            NE1
    Neighborhood_Gilbert            NE2
      Neighborhood_NAmes            NE3
    Neighborhood_NridgHt            NE4
    Neighborhood_OldTown            NE5
      Neighborhood_Sawyer            NE6
    Neighborhood_Somerst            NE7
        Condition1_Feedr             CO
          Condition1_Norm            CO1
            BldgType_1Fam             BL
          BldgType_TwnhsE            BL1
        HouseStyle_1.5Fin             HO
        HouseStyle_1Story            HO1
        HouseStyle_2Story            HO2
          RoofStyle_Gable             RO
      Exterior1st_HdBoard             EX
      Exterior1st_MetalSd            EX1
      Exterior1st_Plywood            EX2
      Exterior1st_VinylSd            EX3
      Exterior1st_Wd Sdng            EX4
      Exterior2nd_HdBoard            EX5
      Exterior2nd_Plywood            EX6
      Exterior2nd_Wd Sdng            EX7
      MasVnrType_BrkFace             MA
            ExterQual_Gd            EX8
            ExterCond_Gd            EX9
            ExterCond_TA           EX10
        Foundation_BrkTil             FO
        Foundation_CBlock            FO1
        Foundation_PConc            FO2
              BsmtQual_Ex            BS1
              BsmtQual_Gd            BS2
              BsmtQual_TA            BS3
              BsmtCond_TA            BS4
          BsmtExposure_Av            BS5
          BsmtExposure_Gd            BS6
          BsmtExposure_Mn            BS7
          BsmtExposure_No            BS8
        BsmtFinType1_ALQ            BS9
        BsmtFinType1_BLQ           BS10
        BsmtFinType1_GLQ           BS11
        BsmtFinType1_LwQ           BS12
        BsmtFinType1_Rec           BS13
        BsmtFinType1_Unf           BS14
        BsmtFinType2_Unf           BS15
            HeatingQC_Ex             HE
            HeatingQC_Gd            HE1
            HeatingQC_TA            HE2
        Electrical_FuseA             EL
        Electrical_SBrkr            EL1
          KitchenQual_Ex             KI
          KitchenQual_Gd            KI1
          KitchenQual_TA            KI2
          Functional_Typ             FU
          FireplaceQu_Gd            FI1
          FireplaceQu_TA            FI2
        GarageType_Attchd            GA3
      GarageType_BuiltIn            GA4
        GarageType_Detchd            GA5
          GarageType_None            GA6
        GarageFinish_Fin            GA7
        GarageFinish_RFn            GA8
        GarageFinish_Unf            GA9
            GarageQual_TA           GA10
            GarageCond_TA           GA11
            PavedDrive_N             PA
            PavedDrive_Y            PA1
              Fence_MnPrv             FE
              Fence_None            FE1
            SaleType_New             SA
              SaleType_WD            SA1
    SaleCondition_Abnorml            SA2
    SaleCondition_Normal            SA3

### Công thức

Lưu ý: Chỉ xáo trộn dữ liệu một lần và đánh giá toàn bộ $m$ mô hình theo cùng thiết lập cross-validation trên tập huấn luyện. Không dùng tập kiểm tra để chọn mô hình.trên tập huấn luyện. Không dùng tập kiểm tra để chọn mô hình.

$$ \text{SalePrice} = 173742.362 + 701.566 * (MS) + 234.750 * (LO) + 4683.901 * (LO1) + 11328.903 * (OV) + 6338.442 * (OV1) + 8791.904 * (YE) + 1963.934 * (YE1) + 598.200 * (LO2) + 7361.904 * (BS) + 7833.617 * (LO3) + 232.367 * (LO4) + 13452.358 * (GR) - 3970.955 * (BE) + 608.423 * (TO) + 1565.645 * (FI) - 1024.242 * (GA) + 2805.846 * (GA1) + 3427.796 * (GA2) + 1268.335 * (LO5) + 53.223 * (LO6) + 938.161 * (LO7) + 1082.483 * (LO8) + 415.407 * (MO) - 101.043 * (YR) + 1513.266 * (TO1) + 13279.338 * (TO2) + 2173.308 * (MS1) - 214.225 * (MS2) + 336.219 * (AL) + 198.379 * (LO9) - 967.435 * (LA) + 2373.045 * (LO10) + 2225.235 * (LO11) + 2125.918 * (LO12) - 1940.454 * (NE) - 2529.281 * (NE1) - 2190.759 * (NE2) - 2506.525 * (NE3) + 3280.020 * (NE4) - 2515.795 * (NE5) - 658.532 * (NE6) + 2481.152 * (NE7) + 526.713 * (CO) + 3427.398 * (CO1) + 7193.825 * (BL) + 1975.535 * (BL1) + 1210.125 * (HO) + 2564.865 * (HO1) + 3434.652 * (HO2) - 263.516 * (RO) - 4352.500 * (EX) - 1149.586 * (EX1) - 3249.074 * (EX2) - 2623.470 * (EX3) - 3321.004 * (EX4) + 1134.351 * (EX5) - 53.591 * (EX6) + 1054.841 * (EX7) - 390.274 * (MA) + 487.824 * (EX8) - 2396.533 * (EX9) - 1452.083 * (EX10) - 207.982 * (FO) + 296.764 * (FO1) + 1962.628 * (FO2) + 2645.783 * (BS1) - 3846.218 * (BS2) - 2706.544 * (BS3) + 657.129 * (BS4) - 7186.565 * (BS5) - 1376.051 * (BS6) - 5797.773 * (BS7) - 10456.643 * (BS8) - 4996.130 * (BS9) - 4055.171 * (BS10) - 3631.259 * (BS11) - 3847.375 * (BS12) - 4269.896 * (BS13) - 5786.697 * (BS14) + 7294.607 * (BS15) + 150.295 * (HE) - 642.929 * (HE1) - 1249.077 * (HE2) + 117.692 * (EL) - 128.968 * (EL1) + 5417.555 * (KI) + 434.594 * (KI1) - 412.234 * (KI2) + 3426.652 * (FU) + 454.174 * (FI1) + 725.531 * (FI2) + 3125.328 * (GA3) + 2524.982 * (GA4) + 4175.751 * (GA5) + 3858.674 * (GA6) - 88.908 * (GA7) - 1382.608 * (GA8) - 465.740 * (GA9) - 239.338 * (GA10) + 1117.249 * (GA11) + 1313.594 * (PA) + 1966.926 * (PA1) + 888.634 * (FE) + 1032.547 * (FE1) + 4091.926 * (SA) - 1286.633 * (SA1) - 1530.480 * (SA2) + 1742.323 * (SA3)$$